# sm'

In [ ]:
///- --real --test static --symbol

:< testing

In [ ]:
///- --test static --symbol

:< testing

In [ ]:
:< rust_operators

:< sm'_real
:< unsupported

## rust

### std_string_real

In [ ]:
///- --real

:! std_string_real =
    `(
        rust::rust_type
            `($'std_string_String')
            `($'std::string::String')
            "type std_string_String = string"
    )

In [ ]:
:== std_string = std_string_real

:= to_string' \* t. (x : t) : std_string =
    !\($'$"!x.to_string()"')

### from_std_string_real

In [ ]:
///- --real

:= from_std_string_real (str : std_string_real) : string =
    rust::emit_expr `std_string_real `string str ($'"Rc::<str>::from($0)"' : string)

In [ ]:
:= from_std_string (str : std_string) : string =
    backend_switch {
        Rust = \ () => !\\(str, $'"Rc::<str>::from(String::as_str(&$0))"') : string
    }

## sm'

### symbol_to_string_real

In [ ]:
///- --real

:= symbol_to_string_real \* t {symbol}. : string =
    := x = real_core::type_to_symbol `t
    !!!!SymbolToString(x)

### symbol_to_string

In [1]:
///- --real --test --symbol
///> gleam
///> lua
///> fsharp
///> cuda

:< testing
:= x = ::test
:= x = symbol_to_string_real `(`x)
_assert_eq `string "test" x

.gleam (gleam)
__assert_eq { expected = test }
.lua (lua)
__assert_eq { expected = test }
.fsx (fsharp)
__assert_eq { expected = test }
.py (cuda)
__assert_eq { expected = test }

In [ ]:
:= symbol_to_string \* t {symbol}. (x : t) : string =
    ~> symbol_to_string_real `t

In [2]:
///- --test --symbol
///> gleam
///> lua
///> fsharp
///> cuda

::test
|> symbol_to_string
|> _assert_eq "test"

.gleam (gleam)
__assert_eq { expected = test }
.lua (lua)
__assert_eq { expected = test }
.fsx (fsharp)
__assert_eq { expected = test }
.py (cuda)
__assert_eq { expected = test }

In [ ]:
:= index i (str : string) : char =
    i |> sm::index str

:= length \* dim {int}. (input : string) : dim =
    input |> sm::length

#### length (test)

In [3]:
///- --test --symbol
///> gleam
///> fsharp
///> cuda

"abc"
|> length
|> _assert_eq 3i32

.gleam (gleam)
__assert_eq { expected = 3 }
.fsx (fsharp)
__assert_eq { expected = 3 }
.py (cuda)
__assert_eq { expected = 3 }

In [ ]:
:= (~##) s : int =
    s |> length

:= to_char_list (str : string) : list char =
    listm::init (str |> length) \ (i : i64) => str |> index i

#### to_char_list (test)

In [4]:
///- --test --symbol
///> gleam
///> fsharp
///> cuda

"abc"
|> to_char_list
|> _assert_eq [ 'a'; 'b'; 'c' ]

.gleam (gleam)
__assert_eq { expected = Cons(a, Cons(b, Cons(c, Nil))) }
.fsx (fsharp)
__assert_eq { expected = Cons(a, Cons(b, Cons(c, Nil))) }
.py (cuda)
__assert_eq { expected = Cons(a, Cons(b, Cons(c, Nil))) }

In [ ]:
:= to_char_array (str : string) : array_base char =
    am'::init_base (str |> length) \ i => str |> index i

#### to_char_array (test)

In [5]:
///- --test --symbol
///> rust
///> lua
///> gleam
///> cuda

"abc"
|> to_char_array
|> \ x => array x : array _
|> am'::to_list'
|> listm'::unbox
|> _assert_eq [ 'a'; 'b'; 'c' ]

.rs (rust)
__assert_eq { expected = Cons(a, Cons(b, Cons(c, Nil))) }
.lua (lua)
__assert_eq { expected = Cons(a, Cons(b, Cons(c, Nil))) }
.gleam (gleam)
__assert_eq { expected = Cons(a, Cons(b, Cons(c, Nil))) }
.py (cuda)
__assert_eq { expected = Cons(a, Cons(b, Cons(c, Nil))) }

0

In [ ]:
:= is_empty (input : string) : bool =
    (input |> length) = 0i32

:= to_option s =
    ? s |> is_empty
    ?> None
    |: s |> Some

### string_builder_real

In [ ]:
///- --real

:! string_builder_python =
    `(
        global "import io"
        $'' : $'io.StringIO'
    )
:== string_builder_switch =
    {
        Gleam : string
        Fsharp : $'System.Text.StringBuilder'
        Python : string_builder_python
        Rust : $'String'
        TypeScript : string
        Zig : string
    }
:! string_builder_real = $'backend_switch `(string_builder_switch)'

In [ ]:
:== string_builder = string_builder_real

:= string_builder (initial : string) : string_builder =
    := initial =
        ? initial = ""
        ?> ~~ initial
        |: initial

    backend_switch {
        Gleam = \ () => $'!initial ' : string_builder
        Lua = \ () => $'!initial ' : string_builder
        Fsharp = \ () => initial |> $'`string_builder ' : string_builder
        Python = \ () =>
            global "import io"
            global """
                class CustomStringIO(io.StringIO):
                    def __init__(self, init=''):
                        super().__init__()
                        if init != '': self.write(init)
                    def __str__(self): return self.getvalue()
                    def __repr__(self): return self.getvalue()
                """
            $'CustomStringIO(!initial)' : string_builder
        Rust = \ () => $'String::from(&*!initial)' : string_builder
        TypeScript = \ () => $'!initial ' : string_builder
        Zig = \ () => $'!initial ' : string_builder
    }

### append_real

In [ ]:
///- --real

:= append_real (x : string) (s : string) : string =
    !!!!StaticStringConcat(s, x)

In [ ]:
:= append (x : string) (s : string) : string =
    ~> append_real x s

### (++#*)

In [ ]:
///- --real

:= (++#*) (a : string) (b : string) : string =
    append_real b a

In [ ]:
:= (++#) (a : string) (b : string) : string =
    ~> (++#*) a b

:= (++\\#) a b =
    a ++# "\\n" ++# b

:= (++\#) a b =
    a ++# "\n" ++# b

### format_debug_real

In [ ]:
///- --real

:= ^ format_debug_real \* t. (x : t) : string =
    ?: t ?|
    | string => x
    | _ =>
        backend_switch `string `({}) {
            Gleam =
                (\ () =>
                    ?: t ?|
                    | char => convert::to `(`x) `string x
                    | string_builder_real => convert::to `(`x) `string x
                    | _ =>
                        global "import gleam/string"
                        := x =
                            ?: t ?|
                            | array_base _ => $'!x |> spiral_array_to_list' : t
                            | _ => x
                        $'string.inspect(!x)' : string
                    ) : () -> string
            Lua =
                (\ () =>
                    ?: t ?|
                    | ~a * ~b =>
                        := a, b =
                            ? not (var_is `t x)
                            ?> x
                            |: ($'__v[1]._1' : a), ($'__v[1]._2' : b)
                        := head_str = format_debug_real `a a
                        := tail_str = format_debug_real `b b
                        (++#*) head_str ((++#*) ", " tail_str)
                    | {} =>
                        := _, fields =
                            real_core::record_fold
                                \ { state = separator, acc key value } =>
                                    ", ", acc ++#* separator ++#* format_debug_real `(`value) value
                                ("", "") x
                        fields
                    | array_base _ =>
                        (++#*) ((++#*) "[" ($'table.concat(!x, ", ")' : string)) "]"
                    | i64 => $'spiral_w64.tostring(!x, true)' : string
                    | u64 => $'spiral_w64.tostring(!x, false)' : string
                    | t =>
                        ? real_core::union_type_is `t
                        ?> serialize_union `t x
                        |: $'tostring(!x)       ' : string
                ) : () -> string
            Fsharp = (\ () => $'$"%A{!x}"' : string) : () -> string
            Python =
                (\ () =>
                    $'f"{!x}"' : string
                ) : () -> string
            Rust =
                (\ () =>
                    ?: t ?|
                    | char => $'Rc::<str>::from(format\!("{:?}", !x as char))' : string
                    | optionm'::option' ~a =>
                        ? real_core::prim_type_is `a
                        ?> $'Rc::<str>::from(match &!x { Some(v) => format\!("Some {:?}", v), None => String::from("None") })' : string
                        |:
                            := x = $'Rc::new(RefCell::new(!x.clone().into_iter().collect::<Vec<_>>()))' : array_base a
                            ? !!!!EQ(!!!!ArrayLength(`u64, x), 0u64) ?> "None"
                            |: (++#*) "Some " (format_debug_element_rust `a !!!!ArrayIndex(x, 0u64))
                    | () * ~b =>
                        := _, b = x
                        format_debug_real `b b
                    | ~a * ~b =>
                        := a, b = x
                        (++#*) (format_debug_real `a a) ((++#*) ", " (format_debug_real `b b))
                    | array_base ~el =>
                        ? real_core::prim_type_is `el
                        ?> $'Rc::<str>::from(format\!("{:?}", !x.borrow()))' : string
                        |: format_debug_elements_rust `el x
                    | listm'::list' ~el =>
                        ? real_core::prim_type_is `el
                        ?> format_debug_other_rust `t x
                        |: format_debug_elements_rust `el ($'Rc::new(RefCell::new((*!x).clone()))' : array_base el)
                    | seq::seq' ~el =>
                        ? real_core::prim_type_is `el
                        ?> format_debug_other_rust `t x
                        |: format_debug_elements_rust `el ($'Rc::new(RefCell::new((*!x).clone()))' : array_base el)
                    | {} =>
                        := _, fields =
                            real_core::record_fold
                                \ { state = separator, acc key value } =>
                                    "; ", acc ++#* separator ++#* symbol_to_string_real `(`key) ++#* " = " ++#* format_debug_real `(`value) value
                                ("", "") x
                        "{ " ++#* fields ++#* " }"
                    | _ => format_debug_other_rust `t x
                ) : () -> string
            TypeScript =
                (\ () =>
                    ?: t ?|
                    | char => $'"\'" + String.fromCharCode(!x) + "\'"' : string
                    | optionm'::option' ~a =>
                        ? $'!x === undefined' : bool
                        ?> "None"
                        |: (++#*) "Some " (format_debug_real `a ($'!x ' : a))
                    | () * ~b =>
                        := _, b = x
                        format_debug_real `b b
                    | ~a * ~b =>
                        := a, b = x
                        (++#*) (format_debug_real `a a) ((++#*) ", " (format_debug_real `b b))
                    | array_base ~el =>
                        ? (?: el ?| char => !+ | _ => !-)
                        ?> $'"[" + !x.map((c: number) => "\'" + String.fromCharCode(c) + "\'").join(", ") + "]"' : string
                        |? real_core::prim_type_is `el
                        ?> $'"[" + !x.join(", ") + "]"' : string
                        |: $'String(!x)' : string
                    | {} =>
                        := _, fields =
                            real_core::record_fold
                                \ { state = separator, acc key value } =>
                                    "; ", acc ++#* separator ++#* symbol_to_string_real `(`key) ++#* " = " ++#* format_debug_real `(`value) value
                                ("", "") x
                        "{ " ++#* fields ++#* " }"
                    | _ =>
                        ? real_core::union_type_is `t
                        ?> serialize_union_rust `t x
                        |? real_core::nominal_type_is `t
                        ?>
                            := y = real_core::nominal_strip x
                            format_debug_real `(`y) y
                        |: $'String(!x)' : string
                ) : () -> string
            Zig =
                (\ () =>
                    ?: t ?|
                    | char => $'(std.fmt.allocPrint(spiral_gpa, "\'{c}\'", .{ !x }) catch @@panic("out of memory"))' : string
                    | bool => $'(if (!x) "true" else "false")' : string
                    | () * ~b =>
                        := _, b = x
                        format_debug_real `b b
                    | ~a * ~b =>
                        := a, b = x
                        (++#*) (format_debug_real `a a) ((++#*) ", " (format_debug_real `b b))
                    | {} =>
                        := _, fields =
                            real_core::record_fold
                                \ { state = separator, acc key value } =>
                                    "; ", acc ++#* separator ++#* symbol_to_string_real `(`key) ++#* " = " ++#* format_debug_real `(`value) value
                                ("", "") x
                        "{ " ++#* fields ++#* " }"
                    | _ =>
                        ? real_core::union_type_is `t
                        ?> serialize_union_rust `t x
                        |? real_core::nominal_type_is `t
                        ?>
                            := y = real_core::nominal_strip x
                            format_debug_real `(`y) y
                        |? real_core::prim_type_is `t
                        ?> $'(std.fmt.allocPrint(spiral_gpa, "{d}", .{ !x }) catch @@panic("out of memory"))' : string
                        |: $'(std.fmt.allocPrint(spiral_gpa, "{any}", .{ !x }) catch @@panic("out of memory"))' : string
                ) : () -> string
            Pascal =
                (\ () =>
                    ?: t ?|
                    | bool => $'BoolToStr(!x, \'true\', \'false\')' : string
                    | char => $'string(!x)' : string
                    | f32 => $'FloatToStr(!x)' : string
                    | f64 => $'FloatToStr(!x)' : string
                    | () * ~b =>
                        := _, b = x
                        format_debug_real `b b
                    | ~a * ~b =>
                        := a, b = x
                        (++#*) (format_debug_real `a a) ((++#*) ", " (format_debug_real `b b))
                    | {} =>
                        := _, fields =
                            real_core::record_fold
                                \ { state = separator, acc key value } =>
                                    "; ", acc ++#* separator ++#* symbol_to_string_real `(`key) ++#* " = " ++#* format_debug_real `(`value) value
                                ("", "") x
                        "{ " ++#* fields ++#* " }"
                    | _ =>
                        ? real_core::union_type_is `t
                        ?> serialize_union_rust `t x
                        |? real_core::nominal_type_is `t
                        ?>
                            := y = real_core::nominal_strip x
                            format_debug_real `(`y) y
                        |: $'IntToStr(!x)' : string
                ) : () -> string
            Lean =
                (\ () =>
                    ?: t ?|
                    | char => $'("\'" ++ toString !x ++ "\'")' : string
                    | () * ~b =>
                        := _, b = x
                        format_debug_real `b b
                    | ~a * ~b =>
                        := a, b = x
                        (++#*) (format_debug_real `a a) ((++#*) ", " (format_debug_real `b b))
                    | {} =>
                        := _, fields =
                            real_core::record_fold
                                \ { state = separator, acc key value } =>
                                    "; ", acc ++#* separator ++#* symbol_to_string_real `(`key) ++#* " = " ++#* format_debug_real `(`value) value
                                ("", "") x
                        "{ " ++#* fields ++#* " }"
                    | _ =>
                        ? real_core::union_type_is `t
                        ?> serialize_union_rust `t x
                        |? real_core::nominal_type_is `t
                        ?>
                            := y = real_core::nominal_strip x
                            format_debug_real `(`y) y
                        |: $'toString !x ' : string
                ) : () -> string
            Kotlin =
                (\ () =>
                    ?: t ?|
                    | char => $'("\'" + Char(!x) + "\'")' : string
                    | () * ~b =>
                        := _, b = x
                        format_debug_real `b b
                    | ~a * ~b =>
                        := a, b = x
                        (++#*) (format_debug_real `a a) ((++#*) ", " (format_debug_real `b b))
                    | {} =>
                        := _, fields =
                            real_core::record_fold
                                \ { state = separator, acc key value } =>
                                    "; ", acc ++#* separator ++#* symbol_to_string_real `(`key) ++#* " = " ++#* format_debug_real `(`value) value
                                ("", "") x
                        "{ " ++#* fields ++#* " }"
                    | _ =>
                        ? real_core::union_type_is `t
                        ?> serialize_union_rust `t x
                        |? real_core::nominal_type_is `t
                        ?>
                            := y = real_core::nominal_strip x
                            format_debug_real `(`y) y
                        |: $'(!x).toString()' : string
                ) : () -> string
            Swift =
                (\ () =>
                    ?: t ?|
                    | char => $'("\'" + String(UnicodeScalar(UInt32(!x)) ?? " ") + "\'")' : string
                    | () * ~b =>
                        := _, b = x
                        format_debug_real `b b
                    | ~a * ~b =>
                        := a, b = x
                        (++#*) (format_debug_real `a a) ((++#*) ", " (format_debug_real `b b))
                    | {} =>
                        := _, fields =
                            real_core::record_fold
                                \ { state = separator, acc key value } =>
                                    "; ", acc ++#* separator ++#* symbol_to_string_real `(`key) ++#* " = " ++#* format_debug_real `(`value) value
                                ("", "") x
                        "{ " ++#* fields ++#* " }"
                    | _ =>
                        ? real_core::union_type_is `t
                        ?> serialize_union_rust `t x
                        |? real_core::nominal_type_is `t
                        ?>
                            := y = real_core::nominal_strip x
                            format_debug_real `(`y) y
                        |: $'String(describing: !x)' : string
                ) : () -> string
            C =
                (\ () =>
                    := buffer () =
                        global "#include <stdio.h>"
                        global "#include <stdint.h>"
                        global "#include <stdlib.h>"
                        global "#include <string.h>"
                        global "static char spiral_format_buffer[64];"
                    := signed x : string =
                        buffer ()
                        global """static inline uint32_t spiral_format_i64(long long x) { return (uint32_t)snprintf(spiral_format_buffer, sizeof spiral_format_buffer, "%lld", x) + 1u; }"""
                        $'StringLit(spiral_format_i64((long long)(!x)), spiral_format_buffer)' : string
                    := unsigned x : string =
                        buffer ()
                        global """static inline uint32_t spiral_format_u64(unsigned long long x) { return (uint32_t)snprintf(spiral_format_buffer, sizeof spiral_format_buffer, "%llu", x) + 1u; }"""
                        $'StringLit(spiral_format_u64((unsigned long long)(!x)), spiral_format_buffer)' : string
                    := decimal_point () =
                        global """static inline uint32_t spiral_format_decimal_point(int n) { if (!strpbrk(spiral_format_buffer, ".eni")) { spiral_format_buffer[n++] = '.'; spiral_format_buffer[n++] = '0'; spiral_format_buffer[n] = 0; } return (uint32_t)n + 1u; }"""
                    := float64 (x : f64) : string =
                        buffer ()
                        decimal_point ()
                        global """static inline uint32_t spiral_format_f64(double x) { int n = 0; for (int p = 1; p <= 17; p++) { n = snprintf(spiral_format_buffer, sizeof spiral_format_buffer, "%.*g", p, x); if (strtod(spiral_format_buffer, NULL) == x) break; } return spiral_format_decimal_point(n); }"""
                        $'StringLit(spiral_format_f64(!x), spiral_format_buffer)' : string
                    := float32 (x : f32) : string =
                        buffer ()
                        decimal_point ()
                        global """static inline uint32_t spiral_format_f32(float x) { int n = 0; for (int p = 1; p <= 9; p++) { n = snprintf(spiral_format_buffer, sizeof spiral_format_buffer, "%.*g", p, (double)x); if (strtof(spiral_format_buffer, NULL) == x) break; } return spiral_format_decimal_point(n); }"""
                        $'StringLit(spiral_format_f32(!x), spiral_format_buffer)' : string
                    ?: t ?|
                    | char =>
                        buffer ()
                        global """static inline uint32_t spiral_format_char(int32_t x) { return (uint32_t)snprintf(spiral_format_buffer, sizeof spiral_format_buffer, "'%c'", (int)x) + 1u; }"""
                        $'StringLit(spiral_format_char(!x), spiral_format_buffer)' : string
                    | bool => ? x ?> "true" |: "false"
                    | f64 => float64 x
                    | f32 => float32 x
                    | i8 => signed x
                    | i16 => signed x
                    | i32 => signed x
                    | i64 => signed x
                    | u8 => unsigned x
                    | u16 => unsigned x
                    | u32 => unsigned x
                    | u64 => unsigned x
                    | array_base ~el => format_debug_elements_rust `el x
                    | () * ~b =>
                        := _, b = x
                        format_debug_real `b b
                    | ~a * ~b =>
                        := a, b = x
                        (++#*) (format_debug_real `a a) ((++#*) ", " (format_debug_real `b b))
                    | {} =>
                        := _, fields =
                            real_core::record_fold
                                \ { state = separator, acc key value } =>
                                    "; ", acc ++#* separator ++#* symbol_to_string_real `(`key) ++#* " = " ++#* format_debug_real `(`value) value
                                ("", "") x
                        "{ " ++#* fields ++#* " }"
                    | _ =>
                        ? real_core::union_type_is `t
                        ?> serialize_union_rust `t x
                        |? real_core::nominal_type_is `t
                        ?>
                            := y = real_core::nominal_strip x
                            format_debug_real `(`y) y
                        |: $'StringLit(sizeof("<`t>"), "<`t>")' : string
                ) : () -> string
        }
:& := format_debug_other_rust \* t. (x : t) : string =
    ? real_core::prim_type_is `t
    ?> $'Rc::<str>::from(format\!("{:?}", !x))' : string
    |? real_core::union_type_is `t
    ?> serialize_union_rust `t x
    |? real_core::nominal_type_is `t
    ?>
        := payload \* u. (y : u) : string =
            ?: u ?|
            | {} => format_debug_real `u y
            | ~a -> ~b => $'Rc::<str>::from("<`t>")' : string
            | ~a * ~b => format_debug_real `u y
            | array_base _ => format_debug_real `u y
            | _ =>
                ? real_core::prim_type_is `u || real_core::union_type_is `u || real_core::nominal_type_is `u
                ?> format_debug_real `u y
                |:
                    $'Rc::<str>::from({ struct W<T>(T); trait D { fn s(&self) -> String; } impl<T: std::fmt::Debug> D \
                        for &W<T> { fn s(&self) -> String { format\!("{:?}", self.0) } } trait P { fn s(&self) -> \
                        String; } impl<T> P for W<T> { fn s(&self) -> String { String::from("<`t>") } } \
                        (&&W(&!y)).s() })'
                    : string
        := y = real_core::nominal_strip x
        payload `(`y) y
    |: $'Rc::<str>::from("<`t>")' : string
:& := format_debug_elements_rust \* el. (x : array_base el) : string =
    ~~
        := n = !!!!ArrayLength(`u64, x)
        := i = mut `u64 (real_core::dyn 0u64)
        := s = mut `string (real_core::dyn "")
        loop::while' (\ () => !!!!LT(!!!!LayoutIndex(i), n)) \ () =>
            := j = !!!!LayoutIndex(i)
            := e = format_debug_element_rust `el !!!!ArrayIndex(x, j)
            s <- (? !!!!EQ(j, 0u64) ?> e |: !!!!LayoutIndex(s) ++#* ", " ++#* e)
            i <- !!!!Add(j, 1u64)
        "[" ++#* !!!!LayoutIndex(s) ++#* "]"
    : string
:& := format_debug_element_rust \* el. (x : el) : string =
    ?: el ?|
    | ~a * ~b => "(" ++#* format_debug_real `el x ++#* ")"
    | _ => format_debug_real `el x
:& := serialize_union \* t. (x : t) : string =
    ~~
        $'if type(v0) ~= "table" then return tostring(v0) end' : ()
        real_core::unbox x \ k, v =>
            := tag : string = $'__v[1].tag'
            ?: `v ?|
            | () => tag
            | ~a * ~b =>
                ?: a ?|
                | ~c * ~d =>
                    := head_str = format_debug_real `c ($'__v[1]._1 ' : c)
                    := tail_str = format_debug_real `d ($'__v[1]._2  ' : d)
                    := x = format_debug_real `b ($'__v[1]._3   ' : b)
                    tag ++#* "(" ++#* head_str ++#* ", " ++#* tail_str ++#* ", " ++#* x ++#* ")"
                | _ =>
                    := head_str = format_debug_real `a ($'__v[1]._1    ' : a)
                    := tail_str = format_debug_real `b ($'__v[1]._2     ' : b)
                    (++#*) tag ((++#*) "(" ((++#*) head_str ((++#*) ", " ((++#*) tail_str ")"))))
            | ~a =>
                := arg_str = format_debug_real `a ($'__v[1]._1' : a)
                (++#*) tag ((++#*) "(" ((++#*) arg_str ")"))
    : string
:& := serialize_union_rust \* t. (x : t) : string =
    ~~
        real_core::unbox x \ k, v =>
            := name = symbol_to_string_real `(`k)
            ?: `v ?|
            | () => name
            | _ => (++#*) name ((++#*) "(" ((++#*) (format_debug_real `(`v) v) ")"))
    : string

:= obj_to_string_rust_inner_real \* t. (x : t) : string =
    ?: t ?|
    | {} => format_debug_real `t x
    | ~a * ~b => format_debug_real `t x
    | ~a -> ~b => format_debug_real `t x
    | array_base _ => format_debug_real `t x
    | ~f ~a => format_debug_real `t x
    | _ =>
        ? real_core::prim_type_is `t || real_core::union_type_is `t || real_core::nominal_type_is `t
        ?> format_debug_real `t x
        |: $'Rc::<str>::from(format\!("{}", !x))' : string

:= obj_to_string_rust_real \* t. (x : t) : string =
    ?: t ?|
    | char => $'Rc::<str>::from((!x as char).encode_utf8(&mut [0u8; 4]) as &str)' : string
    | string => x
    | ~f ~a => format_debug_real `t x
    | _ =>
        ? real_core::nominal_type_is `t
        ?>
            := x = real_core::nominal_strip x
            obj_to_string_rust_inner_real `(`x) x
        |: format_debug_real `t x

In [ ]:
:= format_debug \* t. (x : t) : string =
    ~> format_debug_real `t x

:= obj_to_string_rust \* t. (x : t) : string =
    ~> obj_to_string_rust_real `t x
:= obj_to_string_fsharp \* t. (x : t) : string =
    $'!x |> _.ToString()' : string
:= obj_to_string_python \* t. (x : t) : string =
    $'str(!x)' : string
:= obj_to_string_typescript \* t. (x : t) : string =
    $'String(!x)' : string
:= obj_to_string_wasm \* t. (x : t) : string =
    ? (~> ?: t ?| string => !+ | _ => !-) ?> $'!x ' : string
    |: $'backend_switch_has_no_Wasm_arm_in_lib_spiral' : string
:= obj_to_string_zig \* t. (x : t) : string =
    ? (~> ?: t ?| string => !+ | string_builder_real => !+ | _ => !-)
    ?> $'!x ' : string
    |? (~> ?: t ?| f32 => !+ | f64 => !+ | _ => !-)
    ?> $'(std.fmt.allocPrint(spiral_gpa, "{d}", .{ !x }) catch @@panic("out of memory"))' : string
    |: $'(std.fmt.allocPrint(spiral_gpa, "{any}", .{ !x }) catch @@panic("out of memory"))' : string

:= obj_to_string_prim \* t. (x : t) (otherwise : () -> string) : string =
    ?
        ~>
            ?: t ?|
            | bool => !+
            | char => !+
            | i8 => !+
            | i16 => !+
            | i32 => !+
            | i64 => !+
            | u8 => !+
            | u16 => !+
            | u32 => !+
            | u64 => !+
            | _ => !-
    ?> !!!!PrimToString(x)
    |: otherwise ()

:= (~#?*) s =
    s |> format_debug

#### format_debug (test)

In [6]:
///- --test --no-agree "tests each backend's native rendering" --symbol
///> gleam
///> lua
///> fsharp
///> cuda
///> rust

#?*("1", "2", "3")
|> _assert_eq (
    backend_switch {
        Gleam = \ () => "#(\"1\", \"2\", \"3\")" : string
        Lua = \ () => "1, 2, 3" : string
        Fsharp = \ () => "struct (\"1\", \"2\", \"3\")" : string
        Python = \ () => "('1', '2', '3')" : string
        Rust = \ () => "1, 2, 3" : string
    }
)

.gleam (gleam)
__assert_eq { expected = #("1", "2", "3") }
.lua (lua)
__assert_eq { expected = 1, 2, 3 }
.fsx (fsharp)
__assert_eq { expected = struct ("1", "2", "3") }
.py (cuda)
__assert_eq { expected = ('1', '2', '3') }
.rs (rust)
__assert_eq { expected = 1, 2, 3 }

0

#### format_debug_lua_record_fields (test)

In [7]:
///- --test --symbol
///> lua

:= ~a = 5i64
{ a } |> format_debug |> _assert_eq "5"
:= ~w = -9223372036854775807i64
:= ~u = 18446744073709551615u64
{ w; u } |> format_debug |> _assert_eq "-9223372036854775807, 18446744073709551615"

__assert_eq { expected = 5 }
__assert_eq { expected = -9223372036854775807, 18446744073709551615 }

#### format_debug_2 (test)

In [8]:
///- --test --no-agree "tests each backend's native rendering" --symbol
///> gleam
///> lua
///> fsharp
///> cuda
///> rust

#?*{ c = "1"; a = "2"; b = "3" }
|> _assert_eq (
    backend_switch {
        Gleam = \ () => "#(\"1\", \"2\", \"3\")" : string
        Lua = \ () => "1, 2, 3" : string
        Fsharp = \ () => "struct (\"1\", \"2\", \"3\")" : string
        Python = \ () => "('1', '2', '3')" : string
        Rust = \ () => "{ c = 1; a = 2; b = 3 }" : string
    }
)

.gleam (gleam)
__assert_eq { expected = #("1", "2", "3") }
.lua (lua)
__assert_eq { expected = 1, 2, 3 }
.fsx (fsharp)
__assert_eq { expected = struct ("1", "2", "3") }
.py (cuda)
__assert_eq { expected = ('1', '2', '3') }
.rs (rust)
__assert_eq { expected = { c = 1; a = 2; b = 3 } }

0

In [ ]:
:+ unwrap_format option t = \ x =>
    x
    |> optionm::value
    |> format_debug

:+ unwrap_format result t u = \ x =>
    x
    |> resultm::get
    |> format_debug

:= obj_to_string \* t. (x : t) : string =
    ? ~> show_composite_real `t () ?> ~> show_real `t x
    |:
        backend_switch {
            Gleam = \ () => obj_to_string_prim x (\ () => #?*x : string)
            Lua = \ () => obj_to_string_prim x (\ () => #?*x : string)
            Fsharp = \ () => obj_to_string_prim x (\ () => obj_to_string_fsharp x)
            Python = \ () => obj_to_string_prim x (\ () => obj_to_string_python x)
            Rust = \ () => obj_to_string_prim x (\ () => obj_to_string_rust x)
            TypeScript = \ () => obj_to_string_prim x (\ () => obj_to_string_typescript x)
            Zig = \ () => obj_to_string_prim x (\ () => obj_to_string_zig x)
            Lean = \ () => obj_to_string_prim x (\ () => $'toString !x ' : string)
            Wasm = \ () => obj_to_string_prim x (\ () => obj_to_string_wasm x)
            C = \ () => obj_to_string_prim x (\ () => failwith "sm'.obj_to_string / the C backend prints integers, bools and chars only")
            Pascal = \ () => obj_to_string_prim x (\ () => failwith "sm'.obj_to_string / the Pascal backend prints integers, bools and chars only")
        }

#### obj_to_string_on_c_prints_ints_bools_and_chars (test)

In [9]:
///- --test --symbol
///> c

(obj_to_string 42i32, obj_to_string (0i64 - 7), obj_to_string !+, obj_to_string 'x')
|> _assert_eq ("42", "-7", "true", "x")

__assert_eq { expected = (42, -7, true, x) }

#### obj_to_string_on_pascal_prints_ints_bools_and_chars (test)

In [10]:
///- --test --symbol
///> pascal

(obj_to_string 42i32, obj_to_string (0i64 - 7), obj_to_string 18446744073709551615u64, obj_to_string !-, obj_to_string 'x')
|> _assert_eq ("42", "-7", "18446744073709551615", "false", "x")

unsupported on this machine: `dcc32` was not found (nor fpc)

#### obj_to_string_prints_prims_like_c (test)

In [11]:
///- --test --symbol
///> gleam
///> lua
///> lean

(obj_to_string (dyn !+), obj_to_string (dyn 'x'), obj_to_string (dyn (0i64 - 7)), obj_to_string (dyn 255u8))
|> _assert_eq ("true", "x", "-7", "255")

.gleam (gleam)
__assert_eq { expected = (true, x, -7, 255) }
.lua (lua)
__assert_eq { expected = (true, x, -7, 255) }
lean: unavailable on this machine (`lean` was not found)

#### obj_to_string_prints_bools_lowercase (test)

In [12]:
///- --test --symbol
///> cuda
///> rust
///> zig

(obj_to_string (dyn !+), obj_to_string (dyn !-)) |> _assert_eq ("true", "false")

.py (cuda)
__assert_eq { expected = (true, false) }
.rs (rust)
__assert_eq { expected = (true, false) }
.zig (zig)
__assert_eq { expected = (true, false) }

0

In [ ]:
:= (~#?**) s =
    s |> obj_to_string

### concat_list

In [ ]:
:~ ^ concat_pass (separator : string) (input : list string) (paired : list string) : list string =
    ?= input ?|
    | left :: right :: rest => concat_pass separator rest (left ++# separator ++# right :: paired)
    | [ last ] => listm::rev (last :: paired)
    | [] => listm::rev paired

:~ ^ concat_balanced (separator : string) (input : list string) : string =
    ?= input ?|
    | [] => ""
    | [ only ] => only
    | _ => concat_balanced separator (concat_pass separator input [])

:= concat_right_fold separator input =
    (input, { acc = ""; sep = "" })
    ||> listm::foldBack \ (x : string) { acc sep } => {
        acc = x ++# sep ++# acc
        sep = separator
    }
    |> \ { acc } => acc

:= concat_list separator input : string =
    backend_switch {
        Gleam = \ () => concat_balanced separator input
        Lua = \ () => concat_right_fold separator input
        Fsharp = \ () => concat_right_fold separator input
        Python = \ () => concat_right_fold separator input
        Cpp = \ () => concat_right_fold separator input
        CppHost = \ () => concat_right_fold separator input
        Rust = \ () =>
            := items : array_base string = input |> listm'::box |> listm'::to_array'
            $'Rc::<str>::from((*!items.borrow()).join(&*!separator))' : string
        TypeScript = \ () => concat_right_fold separator input
        Zig = \ () => concat_right_fold separator input
        Lean = \ () => concat_right_fold separator input
        C = \ () => concat_right_fold separator input
        Pascal = \ () => concat_right_fold separator input
        Wasm = \ () => concat_right_fold separator input
        Bend = \ () => concat_right_fold separator input
        Kotlin = \ () => concat_right_fold separator input
        Swift = \ () => concat_right_fold separator input
    }

In [13]:
///- --test --print-code --symbol
///> gleam
///> cuda
///> rust
///> typescript

[
    "1"
    "2"
    "3"
]
|> \ x =>
    := code = (x : _) |> concat_list "\n"
    code
    |> _assert_eq """
        1
        2
        3
        """

.gleam (gleam)
__assert_eq { expected = 1
2
3 }
.py (cuda)
__assert_eq { expected = 1
2
3 }
.rs (rust)
__assert_eq { expected = 1
2
3 }
.ts (typescript)
__assert_eq { expected = 1
2
3 }

0

In [ ]:
:= slice \* t {number; int}. (from : t) (to : t) s : string =
    backend_switch {
        Gleam = \ () =>
            sm::slice s { from to } : string
        Lua = \ () => sm::slice s { from to } : string
        C = \ () => sm::slice s { from to } : string
        Fsharp = \ () => sm::slice s { from to } : string
        Python = \ () => sm::slice s { from to } : string
        Rust = \ () => sm::slice s { from to } : string
        TypeScript = \ () => sm::slice s { from to } : string
        Zig = \ () => sm::slice s { from to } : string
        Lean = \ () => sm::slice s { from to } : string
        Pascal = \ () => sm::slice s { from to } : string
        Kotlin = \ () => sm::slice s { from to } : string
    }

#### slice (test)

In [14]:
///- --test --print-code --symbol
///> rust

"abcdef"
|> slice 3i32 3i32
|> _assert_eq "d"

"abcdef"
|> slice 1i32 3i32
|> slice 2i32 2i32
|> _assert_eq "d"

"abcdef"
|> slice 1i32 4i32
|> slice 2i32 3i32
|> slice 1i32 1i32
|> _assert_eq "e"

__assert_eq { expected = d }
__assert_eq { expected = d }
__assert_eq { expected = e }

0

In [15]:
///- --test --symbol
///> gleam
///> lua
///> fsharp
///> cuda
///> pascal

"abcdef"
|> slice 1i32 3i32
|> _assert_eq "bcd"

(~~ "fghij")
|> slice 1i32 3i32
|> _assert_eq "ghi"

"klmno"
|> slice 1i32 (~~ 3i32)
|> _assert_eq "lmn"

(~~ "pqrst")
|> slice 1i32 (~~ 3i32)
|> _assert_eq "qrs"

"uvwxy"
|> slice (~~ 1i32) 3i32
|> _assert_eq "vwx"

(~~ "zabcd")
|> slice (~~ 1i32) 3i32
|> _assert_eq "abc"

"efghi"
|> slice (~~ 1i32) (~~ 3i32)
|> _assert_eq "fgh"

(~~ "efghi")
|> slice (~~ 1i32) (~~ 3i32)
|> _assert_eq "fgh"

"abcdef"
|> slice 3i32 4i32
|> _assert_eq "de"

(~~ "efghi")
|> slice (~~ 3i32) (~~ 4i32)
|> _assert_eq "hi"

"bcde"
|> slice (~~ 0i32) 0i32
|> _assert_eq "b"

.gleam (gleam)
__assert_eq { expected = bcd }
__assert_eq { expected = ghi }
__assert_eq { expected = lmn }
__assert_eq { expected = qrs }
__assert_eq { expected = vwx }
__assert_eq { expected = abc }
__assert_eq { expected = fgh }
__assert_eq { expected = fgh }
__assert_eq { expected = de }
__assert_eq { expected = hi }
__assert_eq { expected = b }
.lua (lua)
__assert_eq { expected = bcd }
__assert_eq { expected = ghi }
__assert_eq { expected = lmn }
__assert_eq { expected = qrs }
__assert_eq { expected = vwx }
__assert_eq { expected = abc }
__assert_eq { expected = fgh }
__assert_eq { expected = fgh }
__assert_eq { expected = de }
__assert_eq { expected = hi }
__assert_eq { expected = b }
.fsx (fsharp)
__assert_eq { expected = bcd }
__assert_eq { expected = ghi }
__assert_eq { expected = lmn }
__assert_eq { expected = qrs }
__assert_eq { expected = vwx }
__assert_eq { expected = abc }
__assert_eq { expected = fgh }
__assert_eq { expected = fgh }
__assert_eq { expected = de }
__assert

### format_pretty_real

In [ ]:
///- --real

:= format_pretty_real \* t. (x : t) : string =
    run_target_args `string `t (\ () => x) \|
        | Rust => \ x =>
            := ^ structural \* u. () : bool =
                ?: u ?|
                | optionm'::option' ~a =>
                    ? real_core::prim_type_is `a
                    ?> !-
                    |: !+
                | listm'::list' ~a =>
                    ? real_core::prim_type_is `a
                    ?> !-
                    |: !+
                | seq::seq' ~a =>
                    ? real_core::prim_type_is `a
                    ?> !-
                    |: !+
                | array_base ~a =>
                    ? real_core::prim_type_is `a
                    ?> !-
                    |: !+
                | ~a * ~b =>
                    ? structural `a ()
                    ?> !+
                    |: structural `b ()
                | {} =>
                    real_core::record_type_fold
                        \ acc key => \* v. => ? acc ?> !+ |: structural `v ()
                        !-
                        `u
                | _ => real_core::union_type_is `u
            ? structural `t ()
            ?> format_debug_real `t x
            |:
                := result = rust::emit_expr `t `std_string_real x ($'"format\!(\\\"{:##?}\\\", $0)"' : string)
                from_std_string_real result
        | _ => \ _ => format_debug_real `t x

In [ ]:
:= format_pretty \* t. (x : t) : string =
    ~> format_pretty_real `t x

:= prim x = ~>
    ?= x ?|
    | (x : i8) | (x : i16) | (x : i32) | (x : i64) => "%d", x
    | (x : u8) | (x : u16) | (x : u32) | (x : u64) => "%u", x
    | (x : f32) | (x : f64) => "%f", x
    | (x : string) => "%s", x
    | (x : char) => "%c", x

### printable

In [ ]:
///- --real

:? printable t : t -> ()

### format_real

In [ ]:
///- --real

:~ format_real \* t. (x : t) : string =
    := empty : string =
        backend_switch `string `({}) {
            Gleam =
                (\ () =>
                    ""
                ) : () -> string
            Lua =
                (\ () =>
                    ""
                ) : () -> string
            Fsharp =
                (\ () =>
                    ~~ ""
                ) : () -> string
            Python =
                (\ () =>
                    ""
                ) : () -> string
            Rust =
                (\ () =>
                    ""
                ) : () -> string
            TypeScript =
                (\ () =>
                    ""
                ) : () -> string
            Zig =
                (\ () =>
                    ""
                ) : () -> string
            Lean =
                (\ () =>
                    ""
                ) : () -> string
            Kotlin =
                (\ () =>
                    ""
                ) : () -> string
            Swift =
                (\ () =>
                    ""
                ) : () -> string
            C =
                (\ () =>
                    ""
                ) : () -> string
            Pascal =
                (\ () =>
                    ""
                ) : () -> string
        }
    := result = mut `string empty
    := ^ write x =
        := p ((a : string), b) =
            := s : string =
                backend_switch `string `({}) {
                    Gleam =
                        (\ () =>
                            ?= b ?|
                            | (_ : string) => b : string
                            | _ => format_debug_real `(`b) b
                        ) : () -> string
                    Lua =
                        (\ () =>
                            ?= b ?|
                            | (_ : f32) | (_ : f64) => float_native::shortest `t b
                            | (_ : string) => b : string
                            | _ => format_debug_real `(`b) b
                        ) : () -> string
                    Fsharp =
                        (\ () =>
                            ?= b ?|
                            | (_ : f32) | (_ : f64) => float_native::shortest `t b
                            | (_ : string) => b : string
                            | _ => $'$"{!b}"' : string
                        ) : () -> string
                    Python =
                        (\ () =>
                            ?= b ?|
                            | (_ : f32) | (_ : f64) => float_native::shortest `t b
                            | (_ : string) => b : string
                            | _ => $'f"{!b}"' : string
                        ) : () -> string
                    Rust =
                        (\ () =>
                            ?= b ?|
                            | (_ : f32) | (_ : f64) => float_native::shortest `t b
                            | (_ : string) => b : string
                            | (_ : char) => $'Rc::<str>::from(format\!("{}", !b as char))' : string
                            | _ => $'Rc::<str>::from(format\!("{}", !b))' : string
                        ) : () -> string
                    TypeScript =
                        (\ () =>
                            ?= b ?|
                            | (_ : f32) | (_ : f64) => float_native::shortest `t b
                            | (_ : string) => b : string
                            | (_ : char) => $'String.fromCharCode(!b)' : string
                            | _ => $'String(!b)' : string
                        ) : () -> string
                    Zig =
                        (\ () =>
                            ?= b ?|
                            | (_ : f32) | (_ : f64) => float_native::shortest `t b
                            | (_ : string) => b : string
                            | (_ : char) => $'(std.fmt.allocPrint(spiral_gpa, "{c}", .{ !b }) catch @@panic("out of memory"))' : string
                            | (_ : bool) => ? b ?> "true" |: "false"
                            | _ => $'(std.fmt.allocPrint(spiral_gpa, "{d}", .{ !b }) catch @@panic("out of memory"))' : string
                        ) : () -> string
                    Lean =
                        (\ () =>
                            ?= b ?|
                            | (_ : f32) | (_ : f64) => float_native::shortest `t b
                            | (_ : string) => b : string
                            | _ => $'toString !b ' : string
                        ) : () -> string
                    Kotlin =
                        (\ () =>
                            ?= b ?|
                            | (_ : f32) | (_ : f64) => float_native::shortest `t b
                            | (_ : string) => b : string
                            | (_ : char) => $'Char(!b).toString()' : string
                            | _ => $'(!b).toString()' : string
                        ) : () -> string
                    Swift =
                        (\ () =>
                            ?= b ?|
                            | (_ : f32) | (_ : f64) => float_native::shortest `t b
                            | (_ : string) => b : string
                            | (_ : char) => $'String(UnicodeScalar(UInt32(!b)) ?? " ")' : string
                            | _ => $'String(describing: !b)' : string
                        ) : () -> string
                    C =
                        (\ () =>
                            ?= b ?|
                            | (_ : f32) | (_ : f64) => float_native::shortest `t b
                            | (_ : string) => b : string
                            | (_ : char) =>
                                global "#include <stdio.h>"
                                global "#include <stdint.h>"
                                global "#include <stdlib.h>"
                                global "#include <string.h>"
                                global "static char spiral_format_buffer[64];"
                                global """static inline uint32_t spiral_format_plain_char(int32_t x) { return (uint32_t)snprintf(spiral_format_buffer, sizeof spiral_format_buffer, "%c", (int)x) + 1u; }"""
                                $'StringLit(spiral_format_plain_char(!b), spiral_format_buffer)' : string
                            | _ => format_debug_real `(`b) b
                        ) : () -> string
                    Pascal =
                        (\ () =>
                            ?= b ?|
                            | (_ : f32) | (_ : f64) => float_native::shortest `t b
                            | (_ : string) => b : string
                            | (_ : char) => $'string(!b)' : string
                            | _ => format_debug_real `(`b) b
                        ) : () -> string
                }
            := s : string =
                ?= b ?|
                | (_ : f32) | (_ : f64) => float_text::float_text_repr s
                | _ => s
            backend_switch `(()) `({}) {
                Gleam =
                    (\ () =>
                        exec_unit ((\ () => result <- append_real s ((~*) `string result)) : () -> ())
                    ) : () -> ()
                Lua =
                    (\ () =>
                        exec_unit ((\ () => result <- append_real s ((~*) `string result)) : () -> ())
                    ) : () -> ()
                Fsharp =
                    (\ () =>
                        ~~
                            result <- append_real s ((~*) `string result)
                    ) : () -> ()
                Python =
                    (\ () =>
                        ~~
                            result <- append_real s ((~*) `string result)
                    ) : () -> ()
                Rust =
                    (\ () =>
                        ~~
                            result <- append_real s ((~*) `string result)
                    ) : () -> ()
                TypeScript =
                    (\ () =>
                        ~~
                            result <- append_real s ((~*) `string result)
                    ) : () -> ()
                Zig =
                    (\ () =>
                        ~~
                            result <- append_real s ((~*) `string result)
                    ) : () -> ()
                Lean =
                    (\ () =>
                        ~~
                            result <- append_real s ((~*) `string result)
                    ) : () -> ()

                Kotlin =
                    (\ () =>
                        ~~
                            result <- append_real s ((~*) `string result)
                    ) : () -> ()
                Swift =
                    (\ () =>
                        ~~
                            result <- append_real s ((~*) `string result)
                    ) : () -> ()
                C =
                    (\ () =>
                        ~~
                            result <- append_real s ((~*) `string result)
                    ) : () -> ()
                Pascal =
                    (\ () =>
                        ~~
                            result <- append_real s ((~*) `string result)
                    ) : () -> ()
            }

        ?= x ?|
        | () => ()
        | (x : i8) | (x : i16) | (x : i32) | (x : i64) => p ("%d", x)
        | (x : u8) | (x : u16) | (x : u32) | (x : u64) => p ("%u", x)
        | (x : f32) | (x : f64) => p ("%f", x)
        | (x : string) => p ("%s", x)
        | (x : char) => p ("%c", x)
        | (x : bool) => p ("%s", ? x ?> "true" |: "false")
        | (a, b) => write a . write ", " . write b
        | {} &= x =>
            write "{ "
            := _result =
                real_core::record_fold
                    \ { state = separator key value } =>
                        write separator
                        write (symbol_to_string_real `(`key)) . write " = " . write value
                        "; "
                    () x
            write " }"
        | x ?& real_core::symbol_is x => write (symbol_to_string_real `(`x))
        | x ?& real_core::function_is x => write (x ())
        | x ?& real_core::union_is x =>
            ? real_core::prototype_has `(`x) printable ?> printable `(`x) x
            |:
                := temp () =
                    real_core::unbox x (\ (k, v) =>
                        write k
                        ?= v ?|
                        | () => ()
                        | _ => (~~ write "(" . write v . write ")") : ()
                        )
                write (format_debug_real `(`x) x)
        | x ?& real_core::nominal_is x =>
            ? real_core::prototype_has `(`x) printable ?> printable `(`x) x
            |: write (format_pretty_real `(`x) x)
        | x => write (format_debug_real `(`x) x)
    write x
    (~*) `string result

In [ ]:
:= format \* t. (x : t) : string =
    ~> show_real `t x

#### format (test)

In [16]:
///- --test --symbol
///> gleam
///> lua
///> cuda
///> rust
///> typescript
///> c

("1", "2", [ "3"; "4" ], { b = "5"; c = "6"; a = \ () => "7" }, ;[ 8i32 ])
|> format
|> _assert_eq "(1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = 6; a = 7 }, [8])"

.gleam (gleam)
__assert_eq { expected = (1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = 6; a = 7 }, [8]) }
.lua (lua)
__assert_eq { expected = (1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = 6; a = 7 }, [8]) }
.py (cuda)
__assert_eq { expected = (1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = 6; a = 7 }, [8]) }
.rs (rust)
__assert_eq { expected = (1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = 6; a = 7 }, [8]) }
.ts (typescript)
__assert_eq { expected = (1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = 6; a = 7 }, [8]) }
.c (c)
__assert_eq { expected = (1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = 6; a = 7 }, [8]) }

0

In [ ]:
:= format_fixed_digits (decimals : i32) (x : f64) : string =
    := negative = x < 0.0
    := magnitude = ? negative ?> 0.0 - x |: x
    :~ ^ up (v : f64) (e : i32) : f64 * i32 =
        ? v > 0.0 && v < 4503599627370496.0 ?> up (v * 2.0) (e - 1) |: v, e
    :~ ^ down (v : f64) (e : i32) : f64 * i32 =
        ? v >= 9007199254740992.0 ?> down (v / 2.0) (e + 1) |: v, e
    := raised, raised_by = up magnitude 0
    := mantissa, exponent = down raised raised_by
    := digits : array i32 = create 1200
    :~ ^ clear (i : i32) : () =
        ? i < 1200 ?>
            set digits i 0
            clear (i + 1)
    clear 0
    :~ ^ put (m : u64) (i : i32) : i32 =
        ? m = 0u64
        ?> i
        |:
            set digits i (conv (m % 10u64))
            put (m / 10u64) (i + 1)
    := scale_by (factor : i32) (length : i32) : i32 =
        :~ ^ spill (j : i32) (carry : i32) : i32 =
            ? carry = 0
            ?> j
            |:
                set digits j (carry % 10)
                spill (j + 1) (carry / 10)
        :~ ^ go (i : i32) (carry : i32) : i32 =
            ? i < length
            ?>
                := t = am'::index i digits * factor + carry
                set digits i (t % 10)
                go (i + 1) (t / 10)
            |: spill length carry
        go 0 0
    :~ ^ repeat (times : i32) (factor : i32) (length : i32) : i32 =
        ? times <= 0 ?> length |: repeat (times - 1) factor (scale_by factor length)
    := started = put (conv mantissa) 0
    := scaled = repeat decimals 10 started
    := length, drop =
        ? exponent >= 0 ?> repeat exponent 2 scaled, 0 |: repeat (0 - exponent) 5 scaled, 0 - exponent
    :~ ^ lower_nonzero (i : i32) : bool =
        ? i < 0 ?> !- |? am'::index i digits <> 0 ?> !+ |: lower_nonzero (i - 1)
    := rounds_up =
        ? drop <= 0
        ?> !-
        |:
            := top = am'::index (drop - 1) digits
            := odd = drop < length && am'::index drop digits % 2 = 1
            top > 5 || top = 5 && (lower_nonzero (drop - 2) || odd)
    :~ ^ carry_from (i : i32) (length : i32) : i32 =
        := t = am'::index i digits + 1
        set digits i (t % 10)
        := grown = ? i >= length ?> i + 1 |: length
        ? t < 10 ?> grown |: carry_from (i + 1) grown
    := final = ? rounds_up ?> carry_from drop length |: length
    := top = ? final - drop > decimals + 1 ?> final - 1 |: drop + decimals
    :~ ^ leading (i : i32) : i32 =
        ? i > drop + decimals && am'::index i digits = 0 ?> leading (i - 1) |: i
    := first = leading top
    := digit_text (d : i32) : string =
        ? d = 0 ?> "0" |? d = 1 ?> "1" |? d = 2 ?> "2" |? d = 3 ?> "3" |? d = 4 ?> "4"
        |? d = 5 ?> "5" |? d = 6 ?> "6" |? d = 7 ?> "7" |? d = 8 ?> "8" |: "9"
    :~ ^ text (i : i32) (acc : string) : string =
        ? i < drop
        ?> acc
        |:
            := point = ? decimals > 0 && i = drop + decimals - 1 ?> "." |: ""
            text (i - 1) (acc ++# point ++# digit_text (am'::index i digits))
    (? negative ?> "-" |: "") ++# text first ""

:= format_fixed (decimals : i32) (x : f64) : string =
    backend_switch {
        Fsharp = \ () =>
            $'(!x).ToString("F" + string !decimals, System.Globalization.CultureInfo.InvariantCulture)' : string
        Lua = \ () => $'string.format("%." .. !decimals .. "f", !x)' : string
        Python = \ () => $'format(!x, "." + str(!decimals) + "f")' : string
        Rust = \ () =>
            := precision : convert::unativeint = convert::convert decimals
            $'Rc::<str>::from(format\!("{:.*}", !precision, !x).as_str())' : string
        TypeScript = \ () => $'(!x).toFixed(!decimals)' : string
        C = \ () => format_fixed_digits decimals x
        Gleam = \ () => format_fixed_digits decimals x
        Zig = \ () => format_fixed_digits decimals x
    }

#### format_fixed_avoids_ties (test)

In [17]:
///- --test --symbol
///> rust
///> lua
///> python
///> typescript
///> c
///> gleam
///> zig

(
    2.0 / 3.0 |> format_fixed 3,
    0.0 - 1.0 / 3.0 |> format_fixed 1,
    12.0 |> format_fixed 0,
    0.000000001 |> format_fixed 3,
    1234.5678 |> format_fixed 2
)
|> _assert_eq ("0.667", "-0.3", "12", "0.000", "1234.57")

.rs (rust)
__assert_eq { expected = (0.667, -0.3, 12, 0.000, 1234.57) }
.lua (lua)
__assert_eq { expected = (0.667, -0.3, 12, 0.000, 1234.57) }
.py (python)
__assert_eq { expected = (0.667, -0.3, 12, 0.000, 1234.57) }
.ts (typescript)
__assert_eq { expected = (0.667, -0.3, 12, 0.000, 1234.57) }
.c (c)
__assert_eq { expected = (0.667, -0.3, 12, 0.000, 1234.57) }
.gleam (gleam)
__assert_eq { expected = (0.667, -0.3, 12, 0.000, 1234.57) }
.zig (zig)
__assert_eq { expected = (0.667, -0.3, 12, 0.000, 1234.57) }

0

#### format_fixed_digits_rounds_exact_ties_to_even (test)

In [18]:
///- --test --symbol
///> c
///> gleam
///> zig

(
    0.125 |> format_fixed 2,
    2.5 |> format_fixed 0,
    2.675 |> format_fixed 2,
    9.995 |> format_fixed 2,
    0.0 - 0.04 |> format_fixed 1,
    10000000000000000000000.0 |> format_fixed 2
)
|> _assert_eq ("0.12", "2", "2.67", "9.99", "-0.0", "10000000000000000000000.00")

.c (c)
__assert_eq { expected = (0.12, 2, 2.67, 9.99, -0.0, 10000000000000000000000.00) }
.gleam (gleam)
__assert_eq { expected = (0.12, 2, 2.67, 9.99, -0.0, 10000000000000000000000.00) }
.zig (zig)
__assert_eq { expected = (0.12, 2, 2.67, 9.99, -0.0, 10000000000000000000000.00) }

In [ ]:
:= (~#?) s =
    s |> format

:= (++#?) (a : string) (b : _) : string =
    a ++# #?b

### show_real

In [ ]:
///- --real

:= ^ show_real \* t. (x : t) : string =
    ?: t ?|
    | string => x
    | ~a * ~b => "(" ++#* show_items_real `t x ++#* ")"
    | {} =>
        := _, fields =
            real_core::record_fold
                \ { state = separator, acc key value } =>
                    "; ",
                    acc ++#* separator ++#* symbol_to_string_real `(`key) ++#* " = " ++#* show_real `(`value) value
                ("", "")
                x
        "{ " ++#* fields ++#* " }"
    | array_base ~el => show_elements_real `el x
    | _ =>
        ? real_core::union_type_is `t
        ?> show_union_real `t x
        |? real_core::nominal_type_is `t
        ?> show_nominal_real `t x
        |: format_real `t x
:& := show_items_real \* t. (x : t) : string =
    ?: t ?|
    | () * ~b =>
        := _, b = x
        show_items_real `b b
    | ~a * ~b =>
        := a, b = x
        show_real `a a ++#* ", " ++#* show_items_real `b b
    | _ => show_real `t x
:& := show_union_real \* t. (x : t) : string =
    ~~
        real_core::unbox x \ k, v =>
            := name = symbol_to_string_real `(`k)
            ?: `v ?|
            | () => name
            | _ => name ++#* "(" ++#* show_items_real `(`v) v ++#* ")"
    : string
:& := show_nominal_real \* t. (x : t) : string =
    := y = real_core::nominal_strip x
    ? real_core::prototype_has `t show_as
    ?> show_as `t x
    |? real_core::prototype_has `t printable
    ?> format_real `t x
    |? show_structural_real `(`y) ()
    ?> show_real `(`y) y
    |: format_debug_real `t x
:& := show_structural_real \* u. () : bool =
    ?: u ?|
    | string => !+
    | ~a * ~b => !+
    | {} => !+
    | array_base _ => !+
    | _ => real_core::prim_type_is `u || real_core::union_type_is `u || real_core::nominal_type_is `u
:& := show_elements_real \* el. (x : array_base el) : string =
    ~~
        := n = !!!!ArrayLength(`u64, x)
        := i = mut `u64 (real_core::dyn 0u64)
        := s = mut `string (real_core::dyn "")
        loop::while' (\ () => !!!!LT(!!!!LayoutIndex(i), n)) \ () =>
            := j = !!!!LayoutIndex(i)
            := e = show_real `el !!!!ArrayIndex(x, j)
            s <- (? !!!!EQ(j, 0u64) ?> e |: !!!!LayoutIndex(s) ++#* ", " ++#* e)
            i <- !!!!Add(j, 1u64)
        "[" ++#* !!!!LayoutIndex(s) ++#* "]"
    : string

:= show_composite_real \* t. () : bool =
    ?: t ?|
    | string => !-
    | ~a * ~b => !+
    | {} => !+
    | array_base _ => !+
    | _ => real_core::union_type_is `t

In [ ]:
:= show \* t. (x : t) : string =
    ~> show_real `t x

#### show (test)

In [19]:
///- --test --symbol
///> gleam
///> lua
///> cuda
///> rust
///> typescript

show ("1", 2i32, [ "3"; "4" ], { b = "5"; c = (6i32, Some 'x') }, ;[ ("7", 8i32) ])
|> _assert_eq "(1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = (6, Some(x)) }, [(7, 8)])"

.gleam (gleam)
__assert_eq { expected = (1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = (6, Some(x)) }, [(7, 8)]) }
.lua (lua)
__assert_eq { expected = (1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = (6, Some(x)) }, [(7, 8)]) }
.py (cuda)
__assert_eq { expected = (1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = (6, Some(x)) }, [(7, 8)]) }
.rs (rust)
__assert_eq { expected = (1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = (6, Some(x)) }, [(7, 8)]) }
.ts (typescript)
__assert_eq { expected = (1, 2, Cons(3, Cons(4, Nil)), { b = 5; c = (6, Some(x)) }, [(7, 8)]) }

0

#### show_floats (test)

In [20]:
///- --test --symbol
///> gleam
///> lua
///> cuda
///> rust
///> typescript
///> c
///> zig
///> pascal
///> lean

show (1.5f64, 0.1f64, 2.0f64, 1.5707963267948966f64, -0.25f64)
|> _assert_eq "(1.5, 0.1, 2.0, 1.5707963267948966, -0.25)"

.gleam (gleam)
__assert_eq { expected = (1.5, 0.1, 2.0, 1.5707963267948966, -0.25) }
.lua (lua)
__assert_eq { expected = (1.5, 0.1, 2.0, 1.5707963267948966, -0.25) }
.py (cuda)
__assert_eq { expected = (1.5, 0.1, 2.0, 1.5707963267948966, -0.25) }
.rs (rust)
__assert_eq { expected = (1.5, 0.1, 2.0, 1.5707963267948966, -0.25) }
.ts (typescript)
__assert_eq { expected = (1.5, 0.1, 2.0, 1.5707963267948966, -0.25) }
.c (c)
__assert_eq { expected = (1.5, 0.1, 2.0, 1.5707963267948966, -0.25) }
.zig (zig)
__assert_eq { expected = (1.5, 0.1, 2.0, 1.5707963267948966, -0.25) }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))
lean: unavailable on this machine (`lean` was not found)

0

#### show_floats_in_python_repr_layout (test)

In [21]:
///- --test --symbol
///> gleam
///> lua
///> cuda
///> rust
///> typescript
///> c
///> zig
///> pascal
///> lean

show (
    0.0001f64,
    0.00001f64,
    100.0f64,
    123.45f64,
    1000000000000000.0f64,
    10000000000000000.0f64,
    15000000000000000000000.0f64,
    0.0000001f64,
    0.1f64,
    -2.5f64
)
|> _assert_eq "(0.0001, 1e-05, 100.0, 123.45, 1000000000000000.0, 1e+16, 1.5e+22, 1e-07, 0.1, -2.5)"

.gleam (gleam)
__assert_eq { expected = (0.0001, 1e-05, 100.0, 123.45, 1000000000000000.0, 1e+16, 1.5e+22, 1e-07, 0.1, -2.5) }
.lua (lua)
__assert_eq { expected = (0.0001, 1e-05, 100.0, 123.45, 1000000000000000.0, 1e+16, 1.5e+22, 1e-07, 0.1, -2.5) }
.py (cuda)
__assert_eq { expected = (0.0001, 1e-05, 100.0, 123.45, 1000000000000000.0, 1e+16, 1.5e+22, 1e-07, 0.1, -2.5) }
.rs (rust)
__assert_eq { expected = (0.0001, 1e-05, 100.0, 123.45, 1000000000000000.0, 1e+16, 1.5e+22, 1e-07, 0.1, -2.5) }
.ts (typescript)
__assert_eq { expected = (0.0001, 1e-05, 100.0, 123.45, 1000000000000000.0, 1e+16, 1.5e+22, 1e-07, 0.1, -2.5) }
.c (c)
__assert_eq { expected = (0.0001, 1e-05, 100.0, 123.45, 1000000000000000.0, 1e+16, 1.5e+22, 1e-07, 0.1, -2.5) }
.zig (zig)
__assert_eq { expected = (0.0001, 1e-05, 100.0, 123.45, 1000000000000000.0, 1e+16, 1.5e+22, 1e-07, 0.1, -2.5) }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))
lean: unavailable on this machine (`lean` was not found)

0

In [ ]:
:= to_char_array (str : string) : array_base char =
    arraym::init (str |> length) \ i => str |> index i
    |> \ (array x : array _) => x

#### to_char_array_2 (test)

In [22]:
///- --test --symbol
///> fsharp
///> cuda

"abc"
|> to_char_array
|> format
|> _assert_eq #?(;[ 'a'; 'b'; 'c' ])

.fsx (fsharp)
__assert_eq { expected = [a, b, c] }
.py (cuda)
__assert_eq { expected = [a, b, c] }

In [ ]:
:= concat_array (separator : string) (input : array string) : string =
    run_target \|
        | Rust => \ () =>
            $'Rc::<str>::from((*!input).borrow().iter().map(|x| &**x).collect::<Vec<&str>>().join(&*!separator).as_str\
                ())'
            : string
        | _ => \ () =>
            ({ acc = ""; sep = "" }, input)
            ||> arraym::fold \ { acc sep } (x : string) =>
                    { acc = acc ++# sep ++# x; sep = separator }
            |> \ { acc } => acc

#### concat_array (test)

In [23]:
///- --test --print-code --symbol
///> rust
///> typescript
///> cuda

;[
    "1"
    "2"
    "3"
]
|> \ x =>
    := code = x |> concat_array "\n"
    code
    |> _assert_eq """
        1
        2
        3
        """

.rs (rust)
__assert_eq { expected = 1
2
3 }
.ts (typescript)
__assert_eq { expected = 1
2
3 }
.py (cuda)
__assert_eq { expected = 1
2
3 }

0

#### a_growable_array_joins_in_one_pass (test)

In [24]:
///- --test --symbol
///> rust

:= items : array string = am'::growable ()
am'::push_back "a" items
am'::push_back "" items
am'::push_back "b" items
items |> concat_array "\n" |> _assert_eq "a\n\nb"
(am'::growable () : array string) |> concat_array "," |> _assert_eq ""

__assert_eq { expected = a

b }
__assert_eq { expected =  }

0

In [ ]:
:= ellipsis (max : i32) (s : string) =
    ? sm::length s <= max
    ?> s
    |: s |> slice 0 (max - 1) |> append "..."

#### ellipsis (test)

In [25]:
///- --test --symbol
///> gleam
///> cuda
///> rust
///> typescript

"12345"
|> ellipsis 2
|> _assert_eq "12..."

"12345"
|> ellipsis 4
|> _assert_eq "1234..."

.gleam (gleam)
__assert_eq { expected = 12... }
__assert_eq { expected = 1234... }
.py (cuda)
__assert_eq { expected = 12... }
__assert_eq { expected = 1234... }
.rs (rust)
__assert_eq { expected = 12... }
__assert_eq { expected = 1234... }
.ts (typescript)
__assert_eq { expected = 12... }
__assert_eq { expected = 1234... }

0

In [ ]:
:= clip (width : i32) (s : string) : string =
    run_target \|
        | Rust => \ () =>
            $'{ let text = &*!s; if text.chars().count() <= !width as usize { std::rc::Rc::<str>::from(text) } else { std::rc::Rc::<str>::from(text.chars().take((!width as usize).saturating_sub(3)).collect::<String>() + "...") } }' : string
        | Fsharp => \ () =>
            := count = $'(!s).Length' : i32
            ? count <= width ?> s
            |:
                := keep = ? width > 3i32 ?> width - 3i32 |: 0i32
                := head = $'(!s).Substring(0, !keep)' : string
                head ++# "..."
        | _ => \ () =>
            := s = dyn s
            ? length s <= width ?> s |: s |> slice 0i32 (width - 4i32) |> append "..."

#### clip_keeps_short_text_and_cuts_long_text_on_char_boundaries (test)

In [26]:
///- --test --symbol
///> rust

"abc"
|> clip 5i32
|> _assert_eq "abc"

"abcdefgh"
|> clip 6i32
|> _assert_eq "abc..."

"ééééé"
|> clip 4i32
|> _assert_eq "é..."

__assert_eq { expected = abc }
__assert_eq { expected = abc... }
__assert_eq { expected = é... }

0

## fsharp

In [ ]:
:! gleam_dynamic = $'Dynamic'

:= last_index_of_gleam () =
    global "import gleam/bit_array"
    global "import gleam/dynamic.{type Dynamic}"
    global "import gleam/erlang/atom"
    global """
        @external(erlang, "string", "find")
        fn spiral_sm_find_trailing(text: String, search: String, direction: atom.Atom) -> Dynamic

        @external(erlang, "gleam_stdlib", "identity")
        fn spiral_sm_coerce(value: a) -> b
        """

:= gleam_byte_size (s : string) : i32 =
    last_index_of_gleam ()
    $'bit_array.byte_size(bit_array.from_string(!s))' : i32

:= gleam_find_trailing (search : string) (s : string) : gleam_dynamic =
    last_index_of_gleam ()
    $'spiral_sm_find_trailing(!s, !search, atom.create("trailing"))' : gleam_dynamic

:= gleam_is_nomatch (found : gleam_dynamic) : bool =
    $'!found == spiral_sm_coerce(atom.create("nomatch"))' : bool

:= gleam_dynamic_text (found : gleam_dynamic) : string =
    $'spiral_sm_coerce(!found)' : string

:= last_index_of (search : string) (s : string) : i32 =
    := search = dyn search
    run_target \|
        | Zig => \ () => $'(if (std.mem.lastIndexOf(u8, !s, !search)) |i| @@as(i32, @@intCast(i)) else -1)' : i32
        | Rust => \ () =>
            ($'!s.rfind(&*!search)' : optionm'::option' convert::unativeint)
            |> optionm'::map \ i => convert::convert i
            |> optionm'::unwrap_or -1i32
        | Gleam => \ () =>
            ? search = "" ?> gleam_byte_size s
            |:
                := found = gleam_find_trailing search s
                ? gleam_is_nomatch found ?> -1i32 |: gleam_byte_size s - gleam_byte_size (gleam_dynamic_text found)
        | _ => \ () => $'!s.LastIndexOf !search ' : i32

#### last_index_of_finds_the_last_match_in_bytes (test)

In [27]:
///- --test --symbol
///> rust
///> gleam
///> zig

:= ~s = "abcabc"
(last_index_of "bc" s, last_index_of "x" s, last_index_of "" s) |> _assert_eq (4i32, -1i32, 6i32)

.rs (rust)
__assert_eq { expected = (4, -1, 6) }
.gleam (gleam)
__assert_eq { expected = (4, -1, 6) }
.zig (zig)
__assert_eq { expected = (4, -1, 6) }

0

In [ ]:
:= index_of_char_from (from : i32) (c : char) (s : string) : i32 =
    := len = s |> length
    := ^ body i =
        ? i >= len
        ?> -1i32
        |? c = (s |> index i)
        ?> i - from
        |: 루프 (i + 1i32)
    :& := 루프 i = join_body_unit body (? var_is i ?> i |: len) i
    from |> 루프

#### index_of_char_from (test)

In [28]:
///- --test --symbol

"abcdefabc"
|> index_of_char_from 2i32 'a'
|> _assert_eq 4i32

__assert_eq { expected = 4 }

In [ ]:
:= portable_matches_at (search : string) (s : string) (at : i32) : bool =
    := m : i32 = sm::length search
    :~ ^ same (k : i32) : bool =
        ? k >= m ?> !+
        |? sm::index s (at + k) = sm::index search k ?> same (k + 1)
        |: !-
    same 0

### portable_index_of

In [ ]:
:= portable_ends_with (value : string) (s : string) : bool =
    := n : i32 = sm::length s
    := m : i32 = sm::length value
    m <= n && portable_matches_at value s (n - m)

:= portable_starts_with (value : string) (s : string) : bool =
    := n : i32 = sm::length s
    := m : i32 = sm::length value
    m <= n && portable_matches_at value s 0

:= portable_index_of (search : string) (s : string) : i32 =
    := n : i32 = sm::length s
    := m : i32 = sm::length search
    :~ ^ scan (at : i32) : i32 =
        ? at + m > n ?> -1i32
        |? portable_matches_at search s at ?> at
        |: scan (at + 1)
    scan 0

:~ ^ portable_replace_runs (old : string) (new : string) (s : string) (n : i32) (m : i32) (start : i32) (at : i32) (acc : list string) : list string =
    ? at + m > n ?> (? start < n ?> slice start (n - 1i32) s :: acc |: acc)
    |? portable_matches_at old s at ?>
        := run = ? start < at ?> slice start (at - 1i32) s :: acc |: acc
        portable_replace_runs old new s n m (at + m) (at + m) (new :: run)
    |: portable_replace_runs old new s n m start (at + 1i32) acc

:= portable_replace (old : string) (new : string) (s : string) : string =
    := n : i32 = sm::length s
    := m : i32 = sm::length old
    ? m = 0i32 ?> s
    |: portable_replace_runs old new s n m 0i32 0i32 [] |> listm::rev |> concat_list ""

:= gleam_split_once_head (search : string) (s : string) : optionm'::option' string =
    global "import gleam/pair"
    global "import gleam/result"
    global "import gleam/string"
    $'option.from_result(result.map(string.split_once(!s, !search), pair.first))' : optionm'::option' string

:= index_of (search : string) (s : string) : int =
    backend_switch {
        Gleam = \ () =>
            ? search = "" ?> 0
            |:
                ?= gleam_split_once_head search s |> optionm'::unbox ?|
                | Some head => gleam_byte_size head
                | None => -1
        Lua = \ () =>
            := found = $'string.find(!s, !search, 1, true)' : optionm'::option' int
            optionm'::default_value' 0 found - 1
        Fsharp = \ () => $'(!s).IndexOf(!search, System.StringComparison.Ordinal)' : int
        Python = \ () =>
            := found = $'(!s).find(!search)' : int
            ? found < 0 ?> -1 |: $'len((!s)[:!found].encode("utf-8", "surrogateescape"))' : int
        Rust = \ () =>
            ($'!s.find(&*!search)' : optionm'::option' convert::unativeint)
            |> optionm'::map \ i => convert::convert i
            |> optionm'::unwrap_or -1
        C = \ () => conv (portable_index_of search s) : int
        Pascal = \ () => conv (portable_index_of search s) : int
        Kotlin = \ () => $'(!s).indexOf(!search).let { i -> if (i < 0) -1 else (!s).substring(0, i).toByteArray(Charsets.UTF_8).size }' : int
        TypeScript = \ () => $'!s.indexOf(!search)' : int
        Zig = \ () => $'(if (std.mem.indexOf(u8, !s, !search)) |i| @@as(`int, @@intCast(i)) else -1)' : int
        Lean = \ () =>
            ? search = "" ?> 0
            |:
                := pieces = $'`int.ofNat ((!s).splitOn !search).length' : int
                := head = $'((!s).splitOn !search).headD ""' : string
                ? pieces <= 1 ?> -1 |: $'`int.ofNat (!head).utf8ByteSize' : int
    }

#### index_of (test)

In [29]:
///- --test --symbol
///> gleam
///> cuda
///> rust
///> typescript
///> zig

"123"
|> index_of "2"
|> _assert_eq 1

.gleam (gleam)
__assert_eq { expected = 1 }
.py (cuda)
__assert_eq { expected = 1 }
.rs (rust)
__assert_eq { expected = 1 }
.ts (typescript)
__assert_eq { expected = 1 }
.zig (zig)
__assert_eq { expected = 1 }

0

#### index_of_counts_utf8_bytes_and_whole_needles (test)

In [30]:
///- --test --symbol
///> gleam
///> rust
///> cuda
///> lua
///> zig

("é/x" |> index_of "/", "abcabc" |> index_of "ca", "abc" |> index_of "z")
|> _assert_eq (2, 2, -1)

.gleam (gleam)
__assert_eq { expected = (2, 2, -1) }
.rs (rust)
__assert_eq { expected = (2, 2, -1) }
.py (cuda)
__assert_eq { expected = (2, 2, -1) }
.lua (lua)
__assert_eq { expected = (2, 2, -1) }
.zig (zig)
__assert_eq { expected = (2, 2, -1) }

0

In [ ]:
:= replicate (n : int) (s : string) : string =
    := ^ body i acc =
        ? i >= n
        ?> acc
        |: 루프 (i + 1) (acc ++# s)
    :& := 루프 i = join_body_unit body n i
    루프 0 ""

#### replicate (test)

In [31]:
///- --test --symbol
///> rust
///> gleam

"12"
|> replicate (~~ 3)
|> _assert_eq "121212"

.rs (rust)
__assert_eq { expected = 121212 }
.gleam (gleam)
__assert_eq { expected = 121212 }

0

In [32]:
///- --test --print-code --symbol
///> rust
///> gleam

"12"
|> replicate 3
|> _assert_eq "121212"

.rs (rust)
__assert_eq { expected = 121212 }
.gleam (gleam)
__assert_eq { expected = 121212 }

0

In [33]:
///- --test --symbol
///> gleam
///> cuda
///> rust
///> typescript

"12"
|> replicate 3
|> _assert_eq "121212"

.gleam (gleam)
__assert_eq { expected = 121212 }
.py (cuda)
__assert_eq { expected = 121212 }
.rs (rust)
__assert_eq { expected = 121212 }
.ts (typescript)
__assert_eq { expected = 121212 }

0

In [ ]:
:= pad_left (total_width : int) (padding_char : char) (s : string) : string =
    := padding = padding_char |> obj_to_string |> replicate (total_width - length s)
    padding ++# s

#### pad_left_pads_with_a_char (test)

In [34]:
///- --test --symbol
///> c
///> fsharp

"7" |> pad_left 3 ' ' |> _assert_eq "  7"

.c (c)
__assert_eq { expected =   7 }
.fsx (fsharp)
__assert_eq { expected =   7 }

In [ ]:
:= pad_right (total_width : int) (padding_char : char) (s : string) : string =
    := padding = padding_char |> obj_to_string |> replicate (total_width - length s)
    s ++# padding

#### pad_right (test)

In [35]:
///- --test --symbol
///> gleam
///> cuda
///> rust
///> typescript

"123"
|> pad_right 6 ' '
|> _assert_eq "123   "

.gleam (gleam)
__assert_eq { expected = 123    }
.py (cuda)
__assert_eq { expected = 123    }
.rs (rust)
__assert_eq { expected = 123    }
.ts (typescript)
__assert_eq { expected = 123    }

0

In [ ]:
:= convert_to_utf32 (c : char) : int =
    backend_switch {
        Gleam = \ () => c |> convert::i32
        Lua = \ () => c |> convert::i32
        Fsharp = \ () =>
            run_target_args' c \|
            | Fsharp => \ c => $'System.Char.ConvertToUtf32 (string !c, 0)' : int
            | _ => \ c => c |> convert::i32
        Python = \ () => $'ord(!c)' : int
        Rust = \ () => $'(!c as `int)' : int
        TypeScript = \ () => $'(!c)' : int
        Zig = \ () => $'@@as(`int, !c)' : int
    }

:= ends_with (value : string) (s : string) : bool =
    backend_switch {
        Gleam = \ () =>
            global "import gleam/string"
            $'!s |> string.ends_with(!value)' : bool
        Fsharp = \ () => $'!s.EndsWith (!value, false, null)' : bool
        Python = \ () => $'!s.endswith(!value)' : bool
        Rust = \ () => $'!s.ends_with(&*!value)' : bool
        TypeScript = \ () => $'!s.endsWith(!value)' : bool
        Lua = \ () => $'!value == "" or !s:sub(-##!value) == !value ' : bool
        Zig = \ () => $'std.mem.endsWith(u8, !s, !value)' : bool
        C = \ () => portable_ends_with value s
        Pascal = \ () => portable_ends_with value s
        Kotlin = \ () => $'(!s).endsWith(!value)' : bool
        Lean = \ () => $'(!s).endsWith !value ' : bool
    }

:= starts_with (value : string) (s : string) : bool =
    backend_switch {
        Gleam = \ () =>
            global "import gleam/string"
            $'!s |> string.starts_with(!value)' : bool
        Lua = \ () => $'!value == "" or !s:sub(1, ##!value) == !value ' : bool
        Fsharp = \ () => $'(!s).StartsWith(!value, System.StringComparison.Ordinal)' : bool
        Python = \ () => $'!s.startswith(!value)' : bool
        Rust = \ () => $'!s.starts_with(&*!value)' : bool
        TypeScript = \ () => $'!s.startsWith(!value)' : bool
        Zig = \ () => $'std.mem.startsWith(u8, !s, !value)' : bool
        C = \ () => portable_starts_with value s
        Pascal = \ () => portable_starts_with value s
        Kotlin = \ () => $'(!s).startsWith(!value)' : bool
        Lean = \ () => $'(!s).startsWith !value ' : bool
    }

:= substring (start : i32) (len : i32) (str : string) : string =
    run_target \|
        | Rust => \ () =>
            := from : convert::unativeint = convert::convert start
            := until : convert::unativeint = convert::convert (start + len)
            $'Rc::<str>::from(&(!str)[!from..!until])' : string
        | Zig => \ () =>
            := until = start + len
            $'(!str)[@@intCast(!start)..@@intCast(!until)]' : string
        | Gleam => \ () => sm::slice str { from = start; to = start + len - 1 }
        | Lua => \ () =>
            := from = start + 1i32
            := until = start + len
            $'string.sub(!str, !from, !until)' : string
        | Pascal => \ () =>
            := from = start + 1i32
            $'Copy(!str, !from, !len)' : string
        | _ => \ () => $'!str.Substring (!start, !len)'

#### substring_cuts_by_start_and_length (test)

In [36]:
///- --test --symbol
///> gleam
///> rust

:= before_slash (s : string) = s |> substring 0 (s |> index_of "/")
(
    "spiral" |> substring 1 3,
    "spiral" |> substring 0 0,
    "spiral" |> substring 6 0,
    before_slash "é/x",
    before_slash "a’😀/b"
)
|> _assert_eq ("pir", "", "", "é", "a’😀")

.gleam (gleam)
__assert_eq { expected = (pir, , , é, a’😀) }
.rs (rust)
__assert_eq { expected = (pir, , , é, a’😀) }

0

In [ ]:
:~ ^ ascii_case_runs (upper : bool) (s : string) (n : i32) (start : i32) (i : i32) (acc : list string) : list string =
    ? i >= n ?> (? start < n ?> slice start (n - 1i32) s :: acc |: acc)
    |:
        := b : u8 = convert::u8 (index i s)
        := hit = ? upper ?> 97u8 <= b && b <= 122u8 |: 65u8 <= b && b <= 90u8
        ? hit ?>
            := run = ? start < i ?> slice start (i - 1i32) s :: acc |: acc
            := at : i32 = convert::i32 (? upper ?> b - 97u8 |: b - 65u8)
            := letter = ? upper ?> slice at at "ABCDEFGHIJKLMNOPQRSTUVWXYZ" |: slice at at "abcdefghijklmnopqrstuvwxyz"
            ascii_case_runs upper s n (i + 1i32) (i + 1i32) (letter :: run)
        |: ascii_case_runs upper s n start (i + 1i32) acc

:= ascii_case (upper : bool) (s : string) : string =
    := n : i32 = length s
    ascii_case_runs upper s n 0i32 0i32 [] |> listm::rev |> concat_list ""

:= to_lower (input : string) : string =
    backend_switch {
        Gleam = \ () =>
            global "import gleam/string"
            $'!input |> string.lowercase' : string
        Lua = \ () => $'string.lower(!input)' : string
        Fsharp = \ () => $'!input.ToLower' () : string
        Python = \ () => $'!input.lower()' : string
        Rust = \ () => $'Rc::<str>::from(!input.to_lowercase())' : string
        TypeScript = \ () => $'!input.toLowerCase()' : string
        Zig = \ () => $'(std.ascii.allocLowerString(spiral_gpa, !input) catch @@panic("out of memory"))' : string
        Lean = \ () => $'(!input).toLower' : string
        Pascal = \ () => $'LowerCase(!input)' : string
        C = \ () => ascii_case !- input
    }

:~ ^ kebab_from (text : string) (size : i32) (at : i32) (out : string) : string =
    ? at >= size ?> out
    |:
        := letter = substring at 1i32 text
        := lower = to_lower letter
        := dash = ? at > 0i32 && letter <> lower ?> "-" |: ""
        kebab_from text size (at + 1i32) (out ++# dash ++# lower)

:= kebab_of (text : string) : string = kebab_from text (length text) 0i32 ""

#### kebab_of_splits_pascal_case_words (test)

In [37]:
///- --test --symbol
///> rust

(kebab_of "WaitClear", kebab_of "Gadt", kebab_of "ABC", kebab_of "")
|> _assert_eq ("wait-clear", "gadt", "a-b-c", "")

__assert_eq { expected = (wait-clear, gadt, a-b-c, ) }

0

In [ ]:
:= to_upper (input : string) : string =
    backend_switch {
        Gleam = \ () =>
            global "import gleam/string"
            $'!input |> string.uppercase' : string
        Lua = \ () => $'string.upper(!input)' : string
        Fsharp = \ () => $'!input.ToUpper' () : string
        Python = \ () => $'!input.upper()' : string
        Rust = \ () => $'Rc::<str>::from(!input.to_uppercase())' : string
        TypeScript = \ () => $'!input.toUpperCase()' : string
        Zig = \ () => $'(std.ascii.allocUpperString(spiral_gpa, !input) catch @@panic("out of memory"))' : string
        Lean = \ () => $'(!input).toUpper' : string
        Pascal = \ () => $'UpperCase(!input)' : string
        C = \ () => ascii_case !+ input
    }

:= char_to_upper (input : char) : char =
    backend_switch {
        Gleam = \ () =>
            global "import gleam/string"
            $'!input |> string.uppercase' : char
        Lua = \ () => $'string.upper(!input)' : char
        Fsharp = \ () => $'System.Char.ToUpper !input ' : char
        Python = \ () => $'!input.upper()' : char
        Rust = \ () => $'!input.to_ascii_uppercase()' : char
        TypeScript = \ () => $'String.fromCharCode(!input).toUpperCase().charCodeAt(0)' : char
        Zig = \ () => $'std.ascii.toUpper(!input)' : char
        Lean = \ () => $'(!input).toUpper' : char
        C = \ () => $'((!input >= 97 && !input <= 122) ? (char)(!input - 32) : !input)' : char
        Pascal = \ () => $'UpCase(!input)' : char
    }

#### char_to_upper_maps_ascii_letters (test)

In [38]:
///- --test --symbol
///> rust
///> zig
///> c
///> pascal

(char_to_upper 'q', char_to_upper 'Q', char_to_upper '5') |> _assert_eq ('Q', 'Q', '5')

.rs (rust)
__assert_eq { expected = (Q, Q, 5) }
.zig (zig)
__assert_eq { expected = (Q, Q, 5) }
.c (c)
__assert_eq { expected = (Q, Q, 5) }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

#### to_upper_and_to_lower_change_ascii_case (test)

In [39]:
///- --test --symbol
///> rust
///> zig
///> pascal
///> c

(to_upper "abC1", to_lower "ABc1") |> _assert_eq ("ABC1", "abc1")

.rs (rust)
__assert_eq { expected = (ABC1, abc1) }
.zig (zig)
__assert_eq { expected = (ABC1, abc1) }
.c (c)
__assert_eq { expected = (ABC1, abc1) }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

In [ ]:
:= char_code (input : char) : u32 =
    backend_switch {
        Fsharp = \ () => $'uint32 !input ' : u32
        Python = \ () => $'ord(!input)' : u32
        Rust = \ () => $'u32::from(!input)' : u32
        TypeScript = \ () => $'!input' : u32
        Lua = \ () =>
            := byte (at : i32) : u32 = $'string.byte(!input, !at)' : u32
            := lead = byte 1i32
            ? lead < 128u32 ?> lead
            |? lead < 224u32 ?> (lead % 32u32) * 64u32 + byte 2i32 % 64u32
            |? lead < 240u32 ?> (lead % 16u32) * 4096u32 + (byte 2i32 % 64u32) * 64u32 + byte 3i32 % 64u32
            |: (lead % 8u32) * 262144u32 + (byte 2i32 % 64u32) * 4096u32 + (byte 3i32 % 64u32) * 64u32 + byte 4i32 % 64u32
        C = \ () => $'((uint32_t)(unsigned char)!input)' : u32
        Zig = \ () => $'@@as(u32, !input)' : u32
        Pascal = \ () => $'LongWord(Ord(!input))' : u32
        Gleam = \ () =>
            global "import gleam/string"
            $'case string.to_utf_codepoints(!input) { [c, ..] -> string.utf_codepoint_to_int(c) _ -> 0 }' : u32
    }

:| ascii_class =
    | AsciiLetter
    | AsciiDigit
    | AsciiSpace
    | AsciiOther

:= ascii_class_of (input : char) : ascii_class =
    := c : u32 = char_code input
    ? (65u32 <= c && c <= 90u32) || (97u32 <= c && c <= 122u32) ?> AsciiLetter
    |? 48u32 <= c && c <= 57u32 ?> AsciiDigit
    |? c = 32u32 || (9u32 <= c && c <= 13u32) ?> AsciiSpace
    |: AsciiOther

:= is_ascii_letter (input : char) : bool = ?= ascii_class_of input ?| AsciiLetter => !+ | _ => !-

:= is_ascii_digit (input : char) : bool = ?= ascii_class_of input ?| AsciiDigit => !+ | _ => !-

:= is_ascii_space (input : char) : bool = ?= ascii_class_of input ?| AsciiSpace => !+ | _ => !-

:= of_char (input : char) : string =
    backend_switch {
        Fsharp = \ () => $'string !input ' : string
        Python = \ () => $'!input' : string
        Rust = \ () =>
            := text = $'char::from(!input).to_string()' : std_string
            from_std_string text
        TypeScript = \ () => $'String.fromCharCode(!input)' : string
        Lua = \ () => $'!input' : string
        Gleam = \ () => $'!input' : string
        C = \ () => !!!!PrimToString(input) : string
    }

#### char_code_reads_bytes (test)

In [40]:
///- --test --symbol
///> rust
///> c
///> zig
///> pascal

char_code 'A' |> _assert_eq 65u32
char_code 'z' |> _assert_eq 122u32

.rs (rust)
__assert_eq { expected = 65 }
__assert_eq { expected = 122 }
.c (c)
__assert_eq { expected = 65 }
__assert_eq { expected = 122 }
.zig (zig)
__assert_eq { expected = 65 }
__assert_eq { expected = 122 }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

#### ascii_class_of_is_one_total_classifier (test)

In [41]:
///- --test --symbol
///> rust
///> c
///> zig
///> pascal
///> python
///> typescript

(is_ascii_letter 'q', is_ascii_letter 'Q', is_ascii_letter '5') |> _assert_eq (!+, !+, !-)
(is_ascii_digit '0', is_ascii_digit '9', is_ascii_digit 'a') |> _assert_eq (!+, !+, !-)
(is_ascii_space ' ', is_ascii_space '\t', is_ascii_space '_') |> _assert_eq (!+, !+, !-)

.rs (rust)
__assert_eq { expected = (true, true, false) }
__assert_eq { expected = (true, true, false) }
__assert_eq { expected = (true, true, false) }
.c (c)
__assert_eq { expected = (true, true, false) }
__assert_eq { expected = (true, true, false) }
__assert_eq { expected = (true, true, false) }
.zig (zig)
__assert_eq { expected = (true, true, false) }
__assert_eq { expected = (true, true, false) }
__assert_eq { expected = (true, true, false) }
.py (python)
__assert_eq { expected = (true, true, false) }
__assert_eq { expected = (true, true, false) }
__assert_eq { expected = (true, true, false) }
.ts (typescript)
__assert_eq { expected = (true, true, false) }
__assert_eq { expected = (true, true, false) }
__assert_eq { expected = (true, true, false) }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

#### char_code_and_of_char (test)

In [42]:
///- --test --symbol
///> rust

char_code 'A' |> _assert_eq 65u32
char_code 'z' |> _assert_eq 122u32
of_char 'A' |> _assert_eq "A"

__assert_eq { expected = 65 }
__assert_eq { expected = 122 }
__assert_eq { expected = A }

0

#### of_char_is_the_one_character_string_on_gleam (test)

In [43]:
///- --test --symbol
///> gleam

(of_char 'A', of_char '(' = "(") |> _assert_eq ("A", !+)

__assert_eq { expected = (A, true) }

#### of_char_is_the_one_character_string_on_c (test)

In [44]:
///- --test --symbol
///> c

:= ~c = 'A'
(of_char c, of_char '(' = "(") |> _assert_eq ("A", !+)

__assert_eq { expected = (A, true) }

#### char_code_is_the_code_point_on_gleam (test)

In [45]:
///- --test --symbol
///> gleam

(char_code 'A', char_code '7', char_code 'é', char_code '€') |> _assert_eq (65u32, 55u32, 233u32, 8364u32)

__assert_eq { expected = (65, 55, 233, 8364) }

#### char_code_decodes_utf8 (test)

In [46]:
///- --test --symbol
///> lua

(char_code 'A', char_code 'é', char_code '€', of_char 'é')
|> _assert_eq (65u32, 233u32, 8364u32, "é")

__assert_eq { expected = (65, 233, 8364, é) }

In [ ]:
:= builder_append (item : string) (sb : string_builder) : string_builder =
    backend_switch {
        Gleam = \ () =>
            $'!sb <> !item ' : string_builder
        Lua = \ () =>
            $'!sb .. !item ' : string_builder
        Fsharp = \ () =>
            ($'!sb.Append' item : string_builder) |> ignore
            sb
        Python = \ () =>
            ($'!sb.write(!item)' : int) |> ignore
            sb
        Rust = \ () => $'[(!sb).as_str(), &*!item].concat()' : string_builder
        TypeScript = \ () => $'!sb + !item ' : string_builder
        Zig = \ () => $'spiralConcat(!sb, !item)' : string_builder
    }

#### builder_append (test)

In [47]:
///- --test --symbol
///> gleam
///> cuda
///> rust
///> typescript

"ab"
|> string_builder
|> builder_append "cd"
|> builder_append "ef"
|> obj_to_string
|> _assert_eq "abcdef"

.gleam (gleam)
__assert_eq { expected = abcdef }
.py (cuda)
__assert_eq { expected = abcdef }
.rs (rust)
__assert_eq { expected = abcdef }
.ts (typescript)
__assert_eq { expected = abcdef }

0

In [ ]:
:= builder_replace (old : string) (new : string) (sb : string_builder) : string_builder =
    backend_switch {
        Gleam = \ () =>
            global "import gleam/string"
            $'!sb |> string.replace(!old, !new)' : string_builder
        Lua = \ () =>
            $'((!sb):gsub(((!old):gsub("%p", "%%%0")), ((!new):gsub("%%", "%%%%"))))' : string_builder
        Fsharp = \ () =>
            ($'!sb.Replace (!old, !new)' : string_builder) |> ignore
            sb
        Python = \ () => ($'!sb.getvalue().replace(!old, !new)' : string) |> string_builder
        Rust = \ () => $'!sb.replace(&*!old, &*!new)' : string_builder
        TypeScript = \ () => $'!sb.split(!old).join(!new)' : string_builder
        Zig = \ () =>
            $'(std.mem.replaceOwned(u8, spiral_gpa, !sb, !old, !new) catch @@panic("out of memory"))' : string_builder
    }

#### builder_replace_treats_text_literally (test)

In [48]:
///- --test --symbol
///> lua
///> typescript

"a.b.c" |> string_builder |> builder_replace "." "%" |> obj_to_string |> _assert_eq "a%b%c"

.lua (lua)
__assert_eq { expected = a%b%c }
.ts (typescript)
__assert_eq { expected = a%b%c }

#### builder_replace (test)

In [49]:
///- --test --symbol
///> gleam
///> cuda
///> rust
///> typescript

"ab"
|> string_builder
|> builder_append "cd"
|> builder_replace "cd" "ef"
|> builder_append "gh"
|> obj_to_string
|> _assert_eq "abefgh"

.gleam (gleam)
__assert_eq { expected = abefgh }
.py (cuda)
__assert_eq { expected = abefgh }
.rs (rust)
__assert_eq { expected = abefgh }
.ts (typescript)
__assert_eq { expected = abefgh }

0

In [ ]:
:= builder_clear (sb : string_builder) : string_builder =
    backend_switch {
        Gleam = \ () =>
            $'let !sb = ""'
            sb
        Lua = \ () =>
            $'!sb = ""'
            sb
        Fsharp = \ () =>
            ($'!sb.Clear' () : string_builder) |> ignore
            sb
        Python = \ () =>
            ($'!sb.truncate(0)' : int) |> ignore
            ($'!sb.seek(0)' : int) |> ignore
            sb
        Rust = \ () => $'String::new()' : string_builder
        TypeScript = \ () => $'""' : string_builder
        Zig = \ () => $'""' : string_builder
    }

#### builder_clear (test)

In [50]:
///- --test --symbol
///> gleam
///> cuda
///> rust
///> typescript

"ab"
|> string_builder
|> builder_append "cd"
|> builder_clear
|> builder_append "gh"
|> obj_to_string
|> _assert_eq "gh"

.gleam (gleam)
__assert_eq { expected = gh }
.py (cuda)
__assert_eq { expected = gh }
.rs (rust)
__assert_eq { expected = gh }
.ts (typescript)
__assert_eq { expected = gh }

0

In [ ]:
:= rust_char_offset (text : string) (i : int) : int =
    $'(!text.char_indices().nth(!i as usize).map_or(!text.len(), |(at, _)| at) as `int)' : int

:= builder_insert (i : int) (s : string) (sb : string_builder) : string_builder =
    backend_switch {
        Gleam = \ () =>
            := sb = #?sb
            := a = sb |> slice 0 (i - 1)
            := b = sb |> slice i (length sb - 1)
            [ a; s; b ] |> concat_list "" |> string_builder
        Lua = \ () =>
            := sb = #?sb
            := a = sb |> slice 0 (i - 1)
            := b = sb |> slice i (length sb - 1)
            [ a; s; b ] |> concat_list "" |> string_builder
        Fsharp = \ () =>
            ($'!sb.Insert (!i, !s)' : string_builder) |> ignore
            sb
        Python = \ () =>
            := sb = sb |> obj_to_string
            $'"".join([!sb[:!i], !s, !sb[!i:]])' |> string_builder
        Rust = \ () =>
            := text = sb |> obj_to_string
            := at = rust_char_offset text i
            [ text |> slice 0 (at - 1); s; text |> slice at (length text - 1) ] |> concat_list "" |> string_builder
        Zig = fun () => $'(std.mem.concat(spiral_gpa, u8, &.{ (!sb)[0..\@intCast(!i)], !s, (!sb)[\@intCast(!i)..] }) \
            catch \@panic("out of memory"))' : string_builder
    }

#### builder_insert (test)

In [51]:
///- --test --symbol
///> gleam
///> cuda
///> rust

"ab"
|> string_builder
|> builder_insert 1 "gh"
|> obj_to_string
|> _assert_eq "aghb"

.gleam (gleam)
__assert_eq { expected = aghb }
.py (cuda)
__assert_eq { expected = aghb }
.rs (rust)
__assert_eq { expected = aghb }

0

#### builder_insert_counts_chars_not_bytes (test)

In [52]:
///- --test --symbol
///> rust

:= ~text = "éb"
text |> string_builder |> builder_insert 1 "gh" |> obj_to_string |> _assert_eq "éghb"

__assert_eq { expected = éghb }

0

In [ ]:
:= span_from pred str i0 =
    := len = str |> length
    := ^ body (i : int) =
        ? i >= len
        ?> i
        |? str |> index i |> pred
        ?> 루프 (i + 1)
        |: i
    :& := 루프 i = join_body_unit body (? var_is i ?> i |: len) i
    i0 |> 루프

:= char_contains (chars : list char) (c : char) : bool =
    chars |> listm'::exists' ((=) c)

:= trim_chars () =
    [ ' '; '\t'; '\r'; '\n' ]

:= trim_start (chars : list char) (input : string) : string =
    := chars = ? chars = [] ?> trim_chars () |: chars
    := len = input |> length
    :~ ^ 루프 (i : int) =
        ? i >= len
        ?> len
        |? input |> index i |> char_contains chars
        ?> 루프 (i + 1)
        |: i
    input |> slice (루프 0) (len - 1)

#### trim_start (test)

In [53]:
///- --test --symbol
///> gleam
///> fsharp
///> cuda

"  abc  "
|> trim_start []
|> _assert_eq "abc  "

"abcde"
|> trim_start []
|> _assert_eq "abcde"

"abcde"
|> trim_start [ 'a' ]
|> _assert_eq "bcde"

"abcde"
|> trim_start [ 'b'; 'a' ]
|> _assert_eq "cde"

"abcde"
|> trim_start [ 'c'; 'b'; 'a' ]
|> _assert_eq "de"

.gleam (gleam)
__assert_eq { expected = abc   }
__assert_eq { expected = abcde }
__assert_eq { expected = bcde }
__assert_eq { expected = cde }
__assert_eq { expected = de }
.fsx (fsharp)
__assert_eq { expected = abc   }
__assert_eq { expected = abcde }
__assert_eq { expected = bcde }
__assert_eq { expected = cde }
__assert_eq { expected = de }
.py (cuda)
__assert_eq { expected = abc   }
__assert_eq { expected = abcde }
__assert_eq { expected = bcde }
__assert_eq { expected = cde }
__assert_eq { expected = de }

In [ ]:
:= trim_end (chars : list char) (input : string) : string =
    := chars = ? chars = [] ?> trim_chars () |: chars
    := len = input |> length
    :~ ^ 루프 (i : int) =
        ? i <= 0
        ?> -1
        |? input |> index (i - 1) |> char_contains chars
        ?> 루프 (i - 1)
        |: i - 1
    input |> slice 0 (루프 len)

#### trim_end (test)

In [54]:
///- --test --symbol
///> rust
///> gleam
///> cuda

"  def  "
|> trim_end []
|> _assert_eq "  def"

"abcde"
|> trim_end []
|> _assert_eq "abcde"

"abcde"
|> trim_end [ 'e' ]
|> _assert_eq "abcd"

"abcde"
|> trim_end [ 'e'; 'd' ]
|> _assert_eq "abc"

"abcde"
|> trim_end [ 'e'; 'd'; 'c' ]
|> _assert_eq "ab"

.rs (rust)
__assert_eq { expected =   def }
__assert_eq { expected = abcde }
__assert_eq { expected = abcd }
__assert_eq { expected = abc }
__assert_eq { expected = ab }
.gleam (gleam)
__assert_eq { expected =   def }
__assert_eq { expected = abcde }
__assert_eq { expected = abcd }
__assert_eq { expected = abc }
__assert_eq { expected = ab }
.py (cuda)
__assert_eq { expected =   def }
__assert_eq { expected = abcde }
__assert_eq { expected = abcd }
__assert_eq { expected = abc }
__assert_eq { expected = ab }

0

In [ ]:
:= trim_with (chars : list char) (input : string) : string =
    input
    |> trim_start chars
    |> trim_end chars

#### trim_with (test)

In [55]:
///- --test --symbol

"  jkl  "
|> trim_with [ ' '; '\t'; '\n' ]
|> _assert_eq "jkl"

__assert_eq { expected = jkl }

In [ ]:
:= trim_end_suffix (suffix : string) (input : string) : string =
    := suffix_length : int = suffix |> length
    :~ ^ loop (text : string) : string =
        ? suffix_length > 0 && (text |> ends_with suffix)
        ?> text |> slice 0 ((text |> length) - suffix_length - 1) |> loop
        |: text
    loop input

#### trim_end_suffix_removes_every_trailing_copy (test)

In [56]:
///- --test --symbol
///> rust
///> gleam

(
    "a.spi.spi" |> trim_end_suffix ".spi",
    "a.spi" |> trim_end_suffix ".rs",
    "abc" |> trim_end_suffix "",
    ".spi" |> trim_end_suffix ".spi"
)
|> _assert_eq ("a", "a.spi", "abc", "")

.rs (rust)
__assert_eq { expected = (a, a.spi, abc, ) }
.gleam (gleam)
__assert_eq { expected = (a, a.spi, abc, ) }

0

In [ ]:
:= trim (input : string) : string =
    input |> trim_with []

#### trim (test)

In [57]:
///- --test --symbol

"  ghi  "
|> trim
|> _assert_eq "ghi"

__assert_eq { expected = ghi }

In [ ]:
:= concat (a : string) (b : seq::seq' string) : string =
    backend_switch {
        Gleam = \ () =>
            global "import gleam/string"
            $'!b |> string.join(!a)' : string
        Lua = \ () => $'table.concat(!b, !a)' : string
        Fsharp = \ () =>
            := a =
                ? a = "\n"
                ?> ~~ a
                |: a
            b |> $'String.concat' a : string
        Python = \ () =>
            $'!a.join(!b)' : string
        Rust = \ () => $'Rc::<str>::from(!b.iter().map(|x| x.to_string()).collect::<Vec<_>>().join(&*!a))' : string
        TypeScript = \ () => $'!b.join(!a)' : string
        Zig = \ () => $'(std.mem.join(spiral_gpa, !a, !b) catch @@panic("out of memory"))' : string
    }

:= length' \* dim. (input : string) : dim =
    backend_switch {
        Gleam = \ () =>
            global "import gleam/bit_array"
            $'bit_array.byte_size(bit_array.from_string(!input))' : dim
        Lua = \ () => $'##!input ' : dim
        Fsharp = \ () => input |> $'String.length' : dim
        Python = \ () => $'len((!input).encode("utf-8", "surrogateescape"))' : dim
        Rust = \ () => $'(!input.len() as `dim)' : dim
        Zig = \ () => $'@@as(`dim, @@intCast(!input.len))' : dim
        Lean = \ () => $'(`dim.ofNat (!input).utf8ByteSize)' : dim
    }

#### length_2 (test)

In [58]:
///- --test --symbol
///> rust
///> cuda

"abc"
|> length'
|> _assert_eq 3i32

.rs (rust)
__assert_eq { expected = 3 }
.py (cuda)
__assert_eq { expected = 3 }

0

#### length_counts_utf8_bytes (test)

In [59]:
///- --test --symbol
///> gleam
///> rust
///> cuda

"é😀"
|> length'
|> _assert_eq 6i32

.gleam (gleam)
__assert_eq { expected = 6 }
.rs (rust)
__assert_eq { expected = 6 }
.py (cuda)
__assert_eq { expected = 6 }

0

### to_string any

In [ ]:
:+ to_string any =
    obj_to_string

:= replace (old : string) (new : string) (s : string) : string =
    backend_switch {
        Gleam = \ () =>
            global "import gleam/string"
            $'!s |> string.replace(!old, !new)' : string
        Lua = \ () =>
            $'((!s):gsub(((!old):gsub("%p", "%%%0")), ((!new):gsub("%%", "%%%%"))))' : string
        Fsharp = \ () =>
            $'(!s).Replace(!old, !new, System.StringComparison.Ordinal)' : string
        Kotlin = \ () =>
            $'(!s).replace(!old, !new)' : string
        Python = \ () =>
            $'!s.replace(!old, !new)' : string
        Rust = \ () =>
            $'Rc::<str>::from(!s.replace(&*!old, &*!new))' : string
        TypeScript = \ () =>
            $'!s.split(!old).join(!new)' : string
        Zig = \ () =>
            $'(std.mem.replaceOwned(u8, spiral_gpa, !s, !old, !new) catch @@panic("out of memory"))' : string
        Pascal = \ () =>
            $'StringReplace(!s, !old, !new, [rfReplaceAll])' : string
        Lean = \ () =>
            $'(!s).replace !old !new ' : string
        C = \ () => portable_replace old new s
    }

#### replace_agrees_on_c (test)

In [60]:
///- --test --symbol
///> c
///> rust
///> zig
///> pascal

("a.b.c" |> replace "." "--", "aaa" |> replace "aa" "b", "x" |> replace "y" "z", "" |> replace "a" "b", "é.é" |> replace "." "") |> _assert_eq ("a--b--c", "ba", "x", "", "éé")

.c (c)
__assert_eq { expected = (a--b--c, ba, x, , éé) }
.rs (rust)
__assert_eq { expected = (a--b--c, ba, x, , éé) }
.zig (zig)
__assert_eq { expected = (a--b--c, ba, x, , éé) }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

#### search_and_replace_treat_text_literally (test)

In [61]:
///- --test --symbol
///> lua
///> rust
///> gleam

(
    "a.b%c" |> index_of ".",
    "a.b%c" |> index_of "%",
    "x" |> index_of "y",
    "a.b.c" |> replace "." "%",
    "a.b" |> starts_with "a."
)
|> _assert_eq (1, 3, -1, "a%b%c", !+)

.lua (lua)
__assert_eq { expected = (1, 3, -1, a%b%c, true) }
.rs (rust)
__assert_eq { expected = (1, 3, -1, a%b%c, true) }
.gleam (gleam)
__assert_eq { expected = (1, 3, -1, a%b%c, true) }

0

In [ ]:
:~ ^ split_matches_from (str : string) (separator : string) (at : i32) (k : i32) : bool =
    ? k >= sm::length separator ?> !+
    |? sm::index str (at + k) = sm::index separator k ?> split_matches_from str separator at (k + 1)
    |: !-

:~ ^ split_next_from (str : string) (separator : string) (at : i32) : i32 =
    ? at + sm::length separator > sm::length str ?> sm::length str
    |? split_matches_from str separator at 0 ?> at
    |: split_next_from str separator (at + 1)

:~ ^ split_count (str : string) (separator : string) (at : i32) (parts : i32) : i32 =
    := hit = split_next_from str separator at
    ? hit >= sm::length str ?> parts |: split_count str separator (hit + sm::length separator) (parts + 1)

:~ ^ split_start_of (str : string) (separator : string) (part : i32) (at : i32) : i32 =
    ? part = 0 ?> at |: split_start_of str separator (part - 1) (split_next_from str separator at + sm::length separator)

:= split_part (str : string) (separator : string) (part : i32) : string =
    := from = split_start_of str separator part 0
    sm::slice str { from = from; to = split_next_from str separator from - 1 }

:= split_portable (separator : string) (str : string) : array_base string =
    ? (sm::length separator : i32) = 0i32
    ?> am'::init_base 1i32 \ _ => str
    |: am'::init_base (split_count str separator 0 1) \ (part : i32) => split_part str separator part

:= split (separator : string) (str : string) : array_base string =
    backend_switch {
        Gleam = \ () =>
            global "import gleam/string"
            ($'!str |> string.split(!separator)' : listm'::list' string)
            |> listm'::to_array'
        Fsharp = \ () => $'!str.Split !separator ' : array_base string
        Lua = \ () => split_portable separator str
        C = \ () => split_portable separator str
        Pascal = \ () => split_portable separator str
        Kotlin = \ () => $'(!str).split(!separator).toTypedArray()' : array_base string
        Python = \ () => $'!str.split(!separator)' : array_base string
        Rust = \ () =>
            $'Rc::new(RefCell::new(!str.split(&*!separator).map(|x| Rc::<str>::from(x)).collect::<Vec<Rc<str>>>()))'
            : array_base string
        TypeScript = \ () => $'!str.split(!separator)' : array_base string
        Zig = fun () => $'blk: { var parts: std.ArrayList([]const u8) = .empty; var it = std.mem.splitSequence(u8, \
            !str, !separator); while (it.next()) |p| parts.append(spiral_gpa, p) catch \@panic("out of memory"); \
            break :blk parts.items; }' : array_base string
    }

#### split_keeps_empty_and_edge_parts (test)

In [62]:
///- --test --symbol
///> rust
///> lua
///> c
///> pascal

:= parts = "a,,c" |> split "," |> array
:= edges = ",a," |> split "," |> array
:= pattern_like = "a%b.c" |> split "%" |> array
(
    am'::length parts,
    am'::index 0 parts,
    am'::index 1 parts,
    am'::index 2 parts,
    am'::length edges,
    am'::index 1 edges,
    am'::index 1 pattern_like
)
|> _assert_eq (3, "a", "", "c", 3, "a", "b.c")

.rs (rust)
__assert_eq { expected = (3, a, , c, 3, a, b.c) }
.lua (lua)
__assert_eq { expected = (3, a, , c, 3, a, b.c) }
.c (c)
__assert_eq { expected = (3, a, , c, 3, a, b.c) }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

In [ ]:
:= line_parts (text : string) : array_base string =
    split "\n" text

:= line_part_count (parts : array_base string) : int =
    := n : int = parts |> array |> am'::length
    ? n > 0 && am'::index_base (n - 1) parts = "" ?> n - 1 |: n

:= line_without_return (line : string) : string =
    := n : int = length line
    ? ends_with "\r" line |> not ?> line
    |? n = 1 ?> ""
    |: slice 0 (n - 2) line

:= nonempty_line_fold \* s. (f : s -> string -> s) (init : s) (text : string) : s =
    := parts = line_parts text
    := count = line_part_count parts
    :~ ^ go (i : int) (acc : s) : s =
        ? i >= count ?> acc
        |:
            := line = parts |> am'::index_base i |> line_without_return
            go (i + 1) (? line = "" ?> acc |: f acc line)
    go 0 init

:= lines (text : string) : array string =
    run_target \|
        | Rust => \ () =>
            $'Rc::new(RefCell::new(!text.lines().map(Rc::<str>::from).collect::<Vec<Rc<str>>>()))' : array string
        | _ => \ () =>
            := parts = line_parts text
            (arraym::init (line_part_count parts) (\ i => parts |> am'::index_base i |> line_without_return) : array string)

:= nonempty_lines (text : string) : array string =
    run_target \|
        | Rust => \ () =>
            nonempty_line_fold (\ acc line => acc |> am'::vec_push line) (am'::new_vec []) text |> am'::from_vec
        | _ => \ () =>
            := parts = line_parts text
            (arraym::init (line_part_count parts) (\ i => parts |> am'::index_base i |> line_without_return) : array string)
            |> arraym::filter (\ line => line <> "")

#### lines_follow_rust_line_rules (test)

In [63]:
///- --test --symbol
///> rust
///> gleam
///> c

:= text = "a\r\n\nb\n"
(am'::length (lines text) : int) |> _assert_eq 3
(am'::length (nonempty_lines text) : int) |> _assert_eq 2
am'::index 0 (lines text) |> _assert_eq "a"
am'::index 1 (nonempty_lines text) |> _assert_eq "b"
(am'::length (lines "") : int) |> _assert_eq 0

.rs (rust)
__assert_eq { expected = 3 }
__assert_eq { expected = 2 }
__assert_eq { expected = a }
__assert_eq { expected = b }
__assert_eq { expected = 0 }
.gleam (gleam)
__assert_eq { expected = 3 }
__assert_eq { expected = 2 }
__assert_eq { expected = a }
__assert_eq { expected = b }
__assert_eq { expected = 0 }
.c (c)
__assert_eq { expected = 3 }
__assert_eq { expected = 2 }
__assert_eq { expected = a }
__assert_eq { expected = b }
__assert_eq { expected = 0 }

0

In [ ]:
:= line_count (text : string) : int =
    text |> line_parts |> line_part_count

:= nonempty_line_count (text : string) : int =
    nonempty_line_fold (\ n _ => n + 1) 0 text

:= line_at (index : int) (text : string) : string =
    := parts = line_parts text
    ? index >= 0 && index < line_part_count parts ?> parts |> am'::index_base index |> line_without_return |: ""

:= nonempty_line_at (index : int) (text : string) : string =
    := _, found =
        nonempty_line_fold (\ (seen, found) line => seen + 1, (? seen = index ?> line |: found)) (0, "") text
    found

#### line_counts_and_picks_scan_without_splitting (test)

In [64]:
///- --test --symbol
///> rust
///> gleam

:= text = "a\r\n\nb\n"
line_count text |> _assert_eq 3
nonempty_line_count text |> _assert_eq 2
line_at 0 text |> _assert_eq "a"
line_at 1 text |> _assert_eq ""
nonempty_line_at 1 text |> _assert_eq "b"
line_at 9 text |> _assert_eq ""

.rs (rust)
__assert_eq { expected = 3 }
__assert_eq { expected = 2 }
__assert_eq { expected = a }
__assert_eq { expected =  }
__assert_eq { expected = b }
__assert_eq { expected =  }
.gleam (gleam)
__assert_eq { expected = 3 }
__assert_eq { expected = 2 }
__assert_eq { expected = a }
__assert_eq { expected =  }
__assert_eq { expected = b }
__assert_eq { expected =  }

0

In [ ]:
:= listed (separator : string) (item : string) (input : string) : bool =
    input |> split separator |> am'::any \ part => trim part = item

#### listed_compares_trimmed_parts (test)

In [65]:
///- --test --symbol
///> rust
///> fsharp
///> gleam

(
    "a | b | c" |> listed "|" "b",
    "a|bc" |> listed "|" "b",
    "" |> listed "|" "b",
    "b" |> listed "|" "b"
)
|> _assert_eq (!+, !-, !-, !+)

.rs (rust)
__assert_eq { expected = (true, false, false, true) }
.fsx (fsharp)
__assert_eq { expected = (true, false, false, true) }
.gleam (gleam)
__assert_eq { expected = (true, false, false, true) }

0

In [ ]:
:= split_string (separator : array_base string) (str : string) : array_base string =
    run_target_args (\ () => str, separator) \|
        | Fsharp => \ str, separator => $'!str.Split (!separator, System.StringSplitOptions.None)'
        | _ => \ str, separator => str |> split ((array separator : array _) |> seq::of_array |> concat (~~ ""))

:= join' (concat : string) (s : array string) : string =
    run_target \|
        | Fsharp => \ () => $'System.String.Join (!concat, !s)' : string
        | _ => \ () => concat_array concat s

#### join'_joins_with_the_separator_on_every_backend (test)

In [66]:
///- --test --symbol --name join'_joins_with_the_separator_on_every_backend
///> rust

;[
    "a"
    "b"
    "c"
]
|> join' ", "
|> _assert_eq "a, b, c"

__assert_eq { expected = a, b, c }

0

In [ ]:
:! encoding = $'System.Text.Encoding'

:= encoding_utf8 () : encoding =
    $'`encoding.UTF8'

:= utf8_get_bytes (s : string) : array u8 =
    run_target \|
        | Rust => \ () => $'Rc::new(RefCell::new(!s.as_bytes().to_vec()))' : array u8
        | Zig => \ () => arraym::init (length s : i32) (\ (i : i32) => convert::u8 (char_code (index i s)))
        | C => \ () => arraym::init (length s : i32) (\ (i : i32) => convert::u8 (char_code (index i s)))
        | Pascal => \ () => arraym::init (length s : i32) (\ (i : i32) => convert::u8 (char_code (index i s)))
        | Cuda => \ () => $'cp.array(list(!s.encode("utf-8")), dtype=cp.uint8)' : array u8
        | TypeScript => \ () => $'Array.from(new TextEncoder().encode(!s))' : array u8
        | _ => \ () => s |> (encoding_utf8 () |> $'_.GetBytes') : array u8

#### utf8_get_bytes_reads_ascii_bytes (test)

In [67]:
///- --test --symbol
///> rust
///> zig
///> c
///> pascal
///> python
///> typescript

:= bytes = utf8_get_bytes "Az"
am'::length bytes |> _assert_eq 2i32
am'::index 0i32 bytes |> _assert_eq 65u8
am'::index 1i32 bytes |> _assert_eq 122u8

.rs (rust)
__assert_eq { expected = 2 }
__assert_eq { expected = 65 }
__assert_eq { expected = 122 }
.zig (zig)
__assert_eq { expected = 2 }
__assert_eq { expected = 65 }
__assert_eq { expected = 122 }
.c (c)
__assert_eq { expected = 2 }
__assert_eq { expected = 65 }
__assert_eq { expected = 122 }
.py (python)
__assert_eq { expected = 2 }
__assert_eq { expected = 65 }
__assert_eq { expected = 122 }
.ts (typescript)
__assert_eq { expected = 2 }
__assert_eq { expected = 65 }
__assert_eq { expected = 122 }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

In [ ]:
:= byte_to_string (format : string) (x : u8) : string =
    $'!x.ToString' format

## rust

In [ ]:
:! str =
    `(
        rust::rust_type
            `($'Str')
            `($'str')
            "type Str = string"
    )

:= chars (x : rust::ref str) : rust::mut' (into_iterator char) =
    !\\(x, $'$"$0.chars()"')

:= byte_slice (s : string) : rust::ref (am'::slice u8) =
    backend_switch {
        Rust = \ () => $'Box::leak((!s).as_bytes().to_vec().into_boxed_slice())' : rust::ref (am'::slice u8)
    }

#### byte_slice_holds_the_utf8_bytes (test)

In [68]:
///- --test --symbol
///> rust

:= byte_count (s : string) : i32 = s |> byte_slice |> am'::slice_to_vec |> am'::vec_len |> convert::i32
(byte_count "abc", byte_count "é") |> _assert_eq (3i32, 2i32)

__assert_eq { expected = (3, 2) }

0

In [ ]:
:! display t =
    `(
        rust::rust_type
            `($'std_fmt_Display<`t>')
            `($'std::fmt::Display<`t>')
            "type std_fmt_Display<'T> = class end"
    )

:! base64_decode_error =
    `(
        rust::rust_type
            `($'base64_DecodeError')
            `($'base64::DecodeError')
            "type base64_DecodeError = class end"
    )

:! borsh_io_error =
    `(
        rust::rust_type
            `($'borsh_io_Error')
            `($'borsh::io::Error')
            "type borsh_io_Error = class end"
    )

:! utf8_error =
    `(
        rust::rust_type
            `($'std_str_Utf8Error')
            `($'std::str::Utf8Error')
            "type std_str_Utf8Error = class end"
    )

:! from_utf8_error =
    `(
        rust::rust_type
            `($'std_string_FromUtf8Error')
            `($'std::string::FromUtf8Error')
            "type std_string_FromUtf8Error = class end"
    )

:! json_value =
    `(
        rust::rust_type
            `($'serde_json_Value')
            `($'serde_json::Value')
            "type serde_json_Value = class end"
    )

:! json_error =
    `(
        rust::rust_type
            `($'serde_json_Error')
            `($'serde_json::Error')
            "type serde_json_Error = class end"
    )

:! serde_wasm_bindgen_error =
    `(
        rust::rust_type
            `($'serde_wasm_bindgen_Error')
            `($'serde_wasm_bindgen::Error')
            "type serde_wasm_bindgen_Error = class end"
    )

:! js_string =
    `(
        rust::rust_type
            `($'js_sys_JsString')
            `($'js_sys::JsString')
            "type js_sys_JsString = class end"
    )

:! os_str =
    `(
        rust::rust_type
            `($'std_ffi_OsStr')
            `($'std::ffi::OsStr')
            "type std_ffi_OsStr = class end"
    )

:! c_str =
    `(
        rust::rust_type
            `($'std_ffi_CStr')
            `($'std::ffi::CStr')
            "type std_ffi_CStr = class end"
    )

:! c_string =
    `(
        rust::rust_type
            `($'std_ffi_CString')
            `($'std::ffi::CString')
            "type std_ffi_CString = class end"
    )

:! os_string =
    `(
        rust::rust_type
            `($'std_ffi_OsString')
            `($'std::ffi::OsString')
            "type std_ffi_OsString = class end"
    )

:= raw_string_literal (s : string) : rust::ref str =
    backend_switch {
        Fsharp = \ () => !\($'"r##\\"" + !s + "\\"##"') : rust::ref str
        Rust = \ () => $'(&*!s)' : rust::ref str
    }

:= raw_string_literal_static (s : string) : rust::static_ref str =
    backend_switch {
        Fsharp = \ () => !\($'"r##\\"" + !s + "\\"##"') : rust::static_ref str
        Rust = \ () => $'Box::leak(String::from(&*!s).into_boxed_str())' : rust::static_ref str
    }

:= (~#) s =
    s |> raw_string_literal

:= (~###) s =
    s |> raw_string_literal_static

:= include_str (path : string) : rust::ref str =
    !\($'"include_str\!(\\\"" + !path + "\\\")"')

:= as_str (s : string) : rust::ref str =
    run_target_args (\ () => s) \|
        | Rust => \ s => !\\(s, $'"&*$0"')
        | _ => convert::unbox

:= from_iter \* (t : * -> *). (s : t char) : std_string =
    !\\(s, $'"String::from_iter($0)"')

:= ref_to_std_string (str : rust::ref str) : std_string =
    run_target_args (\ () => str) \|
        | Rust => \ str => !\\(str, $'"String::from($0)"')
        | _ => convert::unbox

:= cow_to_std_string (str : rust::cow str) : std_string =
    !\\(str, $'"String::from($0)"')

:= to_std_string (s : string) : std_string =
    s |> as_str |> ref_to_std_string

:= as_str_std (s : std_string) : rust::ref str =
    !\\(s, $'"$0.as_str()"')

:= into_boxed_str (s : std_string) : rust::box str =
    !\\(s, $'"$0.into_boxed_str()"')

:= to_os_string (s : rust::ref os_str) : os_string =
    !\\(s, $'"$0.to_os_string()"')

:= new_c_string (s : std_string) : c_string =
    !\\(s, $'"std::ffi::CString::new($0).unwrap()"')

:= os_to_str (s : os_string) : optionm'::option' (rust::ref str) =
    !\\(s, $'"$0.to_str()"')

:= from_os_str_ref s =
    s
    |> to_os_string
    |> os_to_str
    |> optionm'::unwrap
    |> ref_to_std_string
    |> from_std_string

:= format_custom' (f : string) x : std_string =
    run_target \|
        | Rust => \ () =>
            !\\(x, $'"format\!(\\\"" + !f + "\\\", $0)"')
        | _ => \ () => null ()

:= format_debug' x : std_string =
    run_target \|
        | Rust => \ () =>
            !\\(x, $'"format\!(\\\"{:?}\\\", $0)"')
        | _ => \ () => null ()

:= format' x : std_string =
    run_target_args (\ () => x) \|
        | Rust => \ x =>
            !\\(x, $'"format\!(\\\"{}\\\", $0)"')
        | _ => \ _ => null ()

:= format_hex' x : std_string =
    !\\(x, $'"format\!(\\\"{:02x}\\\", $0)"')

:= format'' (format : string) : std_string =
    !\($'@@$"format\!(" + !format + ")"')

:! regex =
    `(
        rust::rust_type
            `($'regex_Regex')
            `($'regex::Regex')
            "type regex_Regex = class end"
    )

:! regex_error =
    `(
        rust::rust_type
            `($'regex_Error')
            `($'regex::Error')
            "type regex_Error = class end"
    )

:= new_regex (pattern : string) : resultm::result' regex regex_error =
    !\\(pattern, $'$"regex::Regex::new(&$0)"')

### captures

In [ ]:
:! regex_captures t =
    `(
        rust::rust_type
            `($'regex_Captures<`t>')
            `($'regex::Captures<`t>')
            "type regex_Captures<'T> = class end"
    )

:! regex_capture_matches =
    `(
        rust::rust_type
            `($'regex_CaptureMatches')
            `($'regex::CaptureMatches')
            "type regex_CaptureMatches = class end"
    )

:! regex_capture_names =
    `(
        rust::rust_type
            `($'regex_CaptureNames')
            `($'regex::CaptureNames')
            "type regex_CaptureNames = class end"
    )

:= regex_capture_names (regex : regex) : regex_capture_names =
    !\\(regex, $'$"$0.capture_names()"')

:! match' =
    `(
        rust::rust_type
            `($'regex_Match')
            `($'regex::Match')
            "type regex_Match = class end"
    )

:= regex_captures_iter (s : rust::static_ref (rust::mut' std_string)) (regex : regex) : regex_capture_matches =
    := regex = regex |> rust::emit
    !\\(regex, $'$"$0.captures_iter(!s)"')

:~ regex_captures (s : string) (regex : regex) : am'::vec (map::hash_map string string) =

    := s = s |> to_std_string
    \ () =>
        := matches =
            := s = s |> rust::new_box |> rust::box_leak
            regex |> regex_captures_iter s

        (!\($'"true; let _regex_captures : Vec<_> = !matches.map(|x| { //"') : bool) |> ignore

        := fn
            (match' : rust::static_ref (rust::mut' (regex_captures rust::static_lifetime)))
            : map::hash_map string string
            =

            := names = regex |> regex_capture_names

            (!\($'"true; let _regex_captures : std::collections::HashMap<_, _> = !names.map(|x| { //"') : bool)
            |> ignore

            := fn (n : string) : pair string string =
                := n' = n |> rust::clone

                new_pair n' !\\(n, $'$"!match'.name(&$0).map(|x| x.as_str()).unwrap_or(\\\"\\\").to_string().into()"')

            (!\\(fn !\($'"x.unwrap_or(\\\"\\\").to_string().into()"'), $'"true; $0 }).collect()"') : bool) |> ignore

            !\($'"_regex_captures"')

        := x =
            !\($'$"x"')
            |> rust::new_box
            |> rust::box_leak

        (!\\(fn x, $'"true; $0 }).collect::<Vec<_>>()"') : bool) |> ignore

        !\($'"_regex_captures"')

    |> rust::capture_move

#### regex_captures (test)

In [69]:
///- --test --symbol
///> rust -d regex

"fable-library-ts\\.(?<a>[\\d.]+)$"
|> new_regex
|> resultm::unwrap'
|> regex_captures "fable_modules/fable-library-ts.4.17.0"
|> am'::vec_map (map::to_vec >> am'::vec_sort_by_key id)
|> format_debug
|> _assert_eq (
    am'::base ;[
        am'::base ;[ "", ""; "a", "4.17.0" ]
        |> am'::to_vec
    ]
    |> am'::to_vec
    |> format_debug
)

__assert_eq { expected = [[("", ""), ("a", "4.17.0")]] }

In [ ]:
:= serialize \* t. (x : t) : resultm::result' std_string json_error =
    !\($'"serde_json::to_string(&!x)"')

:= deserialize \* t. (json : string) : resultm::result' t std_string =
    := json = ~~ json
    := json = json |> as_str
    !\\(json, $'"serde_json::from_str(&$0)"')
    |> resultm::map_error' \ (x : json_error) => x |> format'

:= borsh_serialize \* t. (data : t) : am'::vec u8 =
    (!\($'"true; let mut data = Vec::new()"') : bool) |> ignore
    (!\\(data, $'"true; borsh::BorshSerialize::serialize(&$0, &mut data).unwrap()"') : bool) |> ignore
    !\($'"data"')

:= borsh_deserialize \* t. (data : array_base u8) : resultm::result' t std_string =
    := data = data |> am'::as_slice
    (!\($'"true; let mut !data = !data"') : bool) |> ignore
    := result = !\($'"borsh::BorshDeserialize::deserialize(&mut !data)"')
    result
    |> resultm::map_error' \ (x : borsh_io_error) => x |> format'

:= deserialize_vec (value : json_value) : resultm::result' (am'::vec u8) std_string =
    := value = ~~ value
    !\($'"serde_json::from_value(!value)"')
    |> resultm::map_error' \ (x : json_error) => x |> format'

:= encode_uri_component (s : std_string) : js_string =
    !\($'"js_sys::encode_uri_component(&!s)"')

:= percent_encode_gleam () =
    global """
        @external(erlang, "binary", "bin_to_list")
        fn spiral_sm_utf8_bytes(text: String) -> List(Int)
        """

:= utf8_byte_list (s : string) : listm'::list' u8 =
    backend_switch {
        Gleam = \ () =>
            percent_encode_gleam ()
            $'spiral_sm_utf8_bytes(!s)' : listm'::list' u8
        Rust = \ () => $'Rc::new(!s.bytes().collect())' : listm'::list' u8
    }

:= percent_unreserved (b : u8) : bool =
    (b >= 48u8 && b <= 57u8) || (b >= 65u8 && b <= 90u8) || (b >= 97u8 && b <= 122u8)
    || b = 45u8 || b = 46u8 || b = 95u8 || b = 126u8

:= ascii_text (b : u8) : string =
    := position : int = conv b - 32
    := upper = " !\"#$%&'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    := table = upper ++# "[\\]^_`abcdefghijklmnopqrstuvwxyz{|}~"
    slice position position table

:= hex_digit (n : u8) : string =
    := position : int = conv n
    slice position position "0123456789ABCDEF"

:= percent_byte (b : u8) : string =
    ? percent_unreserved b ?> ascii_text b |: "%" ++# hex_digit (b >>> 4i32) ++# hex_digit (b &&& 15u8)

:= percent_encode (s : string) : string =
    s
    |> utf8_byte_list
    |> listm'::fold' percent_byte []
    |> listm::fold (\ acc piece => piece ++# acc) ""

#### percent_encode_keeps_only_unreserved (test)

In [70]:
///- --test --symbol
///> rust
///> gleam

:= ~text = "a b+c,d!~-._/é"
percent_encode text |> _assert_eq "a%20b%2Bc%2Cd%21~-._%2F%C3%A9"

.rs (rust)
__assert_eq { expected = a%20b%2Bc%2Cd%21~-._%2F%C3%A9 }
.gleam (gleam)
__assert_eq { expected = a%20b%2Bc%2Cd%21~-._%2F%C3%A9 }

0

In [ ]:
:= strip_prefix (prefix : char) (s : std_string) : optionm'::option' (rust::ref str) =
    := s = ~~ s
    $'!s.strip_prefix(char::from(!prefix))' : optionm'::option' (rust::ref str)

:= str_from_utf8 (bytes : rust::ref (am'::slice u8)) : resultm::result' (rust::ref str) utf8_error =
    !\\(bytes, $'"std::str::from_utf8($0)"')

:= string_from_utf8 (bytes : am'::vec u8) : resultm::result' std_string from_utf8_error =
    := bytes = ~~ bytes
    !\\(bytes, $'"std::string::String::from_utf8($0)"')

:! encoding' =
    `(
        rust::rust_type
            `($'encoding_rs_Encoding')
            `($'encoding_rs::Encoding')
            "type encoding_rs_Encoding = class end"
    )

:= encoding_utf8' () : rust::ref encoding' =
    !\($'"encoding_rs::UTF_8"')

### encoding_1252

In [ ]:
:= encoding_1252' () : rust::ref encoding' =
    !\($'"encoding_rs::WINDOWS_1252"')

### encoding_encode

In [ ]:
:= encoding_encode' (encoding : rust::ref encoding') (text : string) : rust::cow (am'::slice u8) =
    !\\((encoding, text), $'"$0.encode(&*$1).0"')

:! windows t =
    `(
        rust::rust_type
            `($'std_slice_Windows<`t>')
            `($'std::slice::Windows<`t>')
            "type std_slice_Windows<'T> = class end"
    )

:= windows (len : convert::unativeint) (source : am'::vec u8) : windows u8 =
    := source = source |> rust::new_box |> rust::box_leak
    := result = !\\(len, $'"<[_]>::windows(!source, $0)"')
    result

:= any \* t. (fn : string -> bool) (source : windows t) : bool =
    (!\($'"true; let mut !source = !source"') : bool) |> ignore
    := fn' x =
        x
        |> str_from_utf8
        |> resultm::unwrap_or' #""
        |> ref_to_std_string
        |> from_std_string
        |> fn
    !\\(fn', $'"!source.any(move |x| $0(x))"')

:= slice_contains (text : string) (source : am'::vec u8) : bool =
    \ () =>
        := source = ~~ source
        source
        |> windows (text |> length |> (\ x => x : i32) |> convert::convert)
        |> any ((=.) text)
    |> \ x => ~~ x ()

:= as_bytes (text : string) : rust::ref (am'::slice u8) =
    := text = ~~ text
    !\($'"!text.as_bytes()"')

:= into_bytes (x : std_string) : am'::vec u8 =
    backend_switch {
        Rust = \ () => $'(!x).as_bytes().to_vec()' : am'::vec u8
    }

## python

In [ ]:
:= encode_utf8 (s : string) : string =
    := encoding = "utf-8"
    backend_switch {
        Python = \ () =>
            $'!s.encode(!encoding)' : string
    }

## sm'

In [ ]:
:= contains (value : string) (s : string) : bool =
    backend_switch {
        Gleam = \ () =>
            global "import gleam/string"
            $'!s |> string.contains(!value)' : bool
        Lua = \ () => $'string.find(!s, !value, 1, true) ~= nil ' : bool
        Fsharp = \ () => $'!s.Contains !value ' : bool
        Python = \ () => $'!value in !s ' : bool
        Rust = \ () => $'!s.contains(&*!value)' : bool
        TypeScript = \ () => $'!s.includes(!value)' : bool
        Zig = \ () => $'(std.mem.indexOf(u8, !s, !value) \!= null)' : bool
        C = \ () => portable_index_of value s >= 0i32
        Pascal = \ () => portable_index_of value s >= 0i32
        Kotlin = \ () => $'(!s).contains(!value)' : bool
        Lean = \ () => $'(!s).contains !value ' : bool
    }

#### searches_agree_on_c (test)

In [71]:
///- --test --symbol
///> c
///> gleam
///> rust
///> pascal
///> zig

(
    "a.b%c" |> index_of ".",
    "a.b%c" |> index_of "%",
    "a.b%c" |> index_of "y",
    "a.b%c" |> index_of "",
    "a.b%c" |> index_of "b%c"
)
|> _assert_eq (1, 3, -1, 0, 2)
(
    "a.b%c" |> contains "%c",
    "a.b%c" |> contains "x",
    "a.b%c" |> starts_with "a.",
    "a.b%c" |> starts_with "b",
    "a.b%c" |> ends_with "%c",
    "a.b%c" |> ends_with "a",
    "" |> ends_with ""
)
|> _assert_eq (!+, !-, !+, !-, !+, !-, !+)

.c (c)
__assert_eq { expected = (1, 3, -1, 0, 2) }
__assert_eq { expected = (true, false, true, false, true, false, true) }
.gleam (gleam)
__assert_eq { expected = (1, 3, -1, 0, 2) }
__assert_eq { expected = (true, false, true, false, true, false, true) }
.rs (rust)
__assert_eq { expected = (1, 3, -1, 0, 2) }
__assert_eq { expected = (true, false, true, false, true, false, true) }
.zig (zig)
__assert_eq { expected = (1, 3, -1, 0, 2) }
__assert_eq { expected = (true, false, true, false, true, false, true) }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

### to_string result t u

In [ ]:
:+ to_string result t u = \ x =>
    ~>
        ?: (t * u) ?|
        | string * string =>
            ?= x ?|
            | Ok x => x
            | Error x => "sm'.to_string result / Error: " ++# x
        | std_string * std_string =>
            ?= x ?|
            | Ok x => from_std_string x
            | Error x => "sm'.to_string result / Error: " ++# from_std_string x
        | _ => obj_to_string `u x

:= format_exception (ex : exn) : string =
    run_target \|
        | Zig => \ () =>
            $'!ex ' : string
        | Fsharp => \ () => $'$"{!ex.GetType ()}: {!ex.Message}"'
        | Rust => \ () => $'!ex ' : string
        | TypeScript => \ () => $'String(!ex)' : string
        | _ => \ () => ex |> format_debug

#### format_exception (test)

In [72]:
///- --test --no-agree "exception text is backend-specific" --symbol
///> cuda
///> rust
///> typescript

\ () => failwith "test"
|> _throws
|> optionm::value
|> format_exception
|> _assert_eq (run_target \|
    | Fsharp _ => \ () => ~~ "System.Exception: test"
    | Cuda _ => \ () => "test"
    | Rust => \ () => "test"
    | TypeScript _ => \ () => "Error: test"
    | _ => \ () => null ()
)

.py (cuda)
__assert_eq { expected = test }
.rs (rust)
__assert_eq { expected = test }
.ts (typescript)
__assert_eq { expected = Error: test }

0

In [ ]:
:= range \* t. (start : am'::range t) (end : am'::range t) s =
    := is_start, start, end =
        := len () : int =
            ~>
                ?: t ?|
                | int => length `t s
                | _ => length' `t s
        ?= start, end ?|
        | Start start, End fn => !+, start, len >> convert::to |> fn
        | End start_fn, End end_fn => !-, start_fn (len >> convert::to), end_fn (len >> convert::to)
    := start' = start |> convert::try_convert |> optionm::value : int
    := end' = end |> convert::try_convert |> optionm::value : int
    := start_is_var = start |> var_is
    := same_fixed = start = end && (start_is_var |> not) && (end |> var_is |> not)
    := start_var_s_fixed = start_is_var && (s |> var_is |> not)
    := trace () =
        [
            "sm'.range"
            " / (s |> var_is): "; s |> var_is |> obj_to_string
            " // start: "; start |> obj_to_string
            " / (start |> var_is): "; start_is_var |> obj_to_string
            " // end: "; end |> obj_to_string
            " / (end |> var_is): "; end |> var_is |> obj_to_string
            " // same_fixed: "; same_fixed |> obj_to_string
            " / start_var_s_fixed: "; start_var_s_fixed |> obj_to_string
        ]
        |> concat_list ""
    backend_switch {
        Gleam = \ () =>
            ()
        Lua = id : () -> ()
        Fsharp = id : () -> ()
        Python = id : () -> ()
        Rust = id : () -> ()
        TypeScript = id : () -> ()
        Zig = id : () -> ()
    }
    s |> slice start' (end' - (? is_start && (same_fixed || start_var_s_fixed) ?> 0 |: 1))

#### range (test)

In [73]:
///- --test --symbol
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

"abcde"
|> range (am'::Start 0i32) (am'::End eval)
|> _assert_eq "abcde"

(~~ "fghij")
|> range (am'::Start 1i32) (am'::End eval)
|> _assert_eq "ghij"

(~~ "klmno")
|> range (am'::Start 1i32) (am'::End \ _ => 1)
|> _assert_eq "l"

"pqrst"
|> range (am'::Start 1i32) (am'::End \ _ => 1)
|> _assert_eq "q"

"uvwxy"
|> range (am'::Start (~~ 1i32)) (am'::End \ _ => 1)
|> _assert_eq "v"

"zabcd"
|> range (am'::Start (~~ 1i32)) (am'::End \ _ => ~~ 1)
|> _assert_eq "a"

(~~ "efghi")
|> range (am'::Start 1i32) (am'::End \ _ => 5)
|> _assert_eq "fghi"

"jklmn"
|> range (am'::End \ x => 1i32) (am'::End \ _ => ~~ 5)
|> _assert_eq "klmn"

"opqrs"
|> range (am'::End \ x => x () - 4i32) (am'::End \ x => x () - 1i32)
|> _assert_eq "pqr"

(~~ "stuvw")
|> range (am'::End \ x => x () - 4i32) (am'::End \ x => x () - 1)
|> _assert_eq "tuv"

(~~ "wxyza")
|> range (am'::Start (~~ 2i32)) (am'::End \ x => x () - 2)
|> _assert_eq "y"

(~~ "bcdef")
|> range (am'::Start (~~ 2i32)) (am'::End \ x => x () - 3)
|> _assert_eq ""

(~~ "ghijk")
|> range (am'::Start (~~ 1i32)) (am'::End \ x => x () - 3)
|> _assert_eq "h"

(~~ "lmnop")
|> range (am'::Start (~~ 1i32)) (am'::End \ x => x () - 4)
|> _assert_eq ""

(~~ "qrstu")
|> range (am'::End \ x => x () - 4i32) (am'::End \ _ => 4)
|> _assert_eq "rst"

"vwxyz"
|> range (am'::End \ x => ~~ x () - 4i32) (am'::End \ _ => 4)
|> _assert_eq "wxy"

.gleam (gleam)
__assert_eq { expected = abcde }
__assert_eq { expected = ghij }
__assert_eq { expected = l }
__assert_eq { expected = q }
__assert_eq { expected = v }
__assert_eq { expected = a }
__assert_eq { expected = fghi }
__assert_eq { expected = klmn }
__assert_eq { expected = pqr }
__assert_eq { expected = tuv }
__assert_eq { expected = y }
__assert_eq { expected =  }
__assert_eq { expected = h }
__assert_eq { expected =  }
__assert_eq { expected = rst }
__assert_eq { expected = wxy }
.fsx (fsharp)
__assert_eq { expected = abcde }
__assert_eq { expected = ghij }
__assert_eq { expected = l }
__assert_eq { expected = q }
__assert_eq { expected = v }
__assert_eq { expected = a }
__assert_eq { expected = fghi }
__assert_eq { expected = klmn }
__assert_eq { expected = pqr }
__assert_eq { expected = tuv }
__assert_eq { expected = y }
__assert_eq { expected =  }
__assert_eq { expected = h }
__assert_eq { expected =  }
__assert_eq { expected = rst }
__assert_eq { expected = wxy }
.py (

0

In [ ]:
:= decimal_text \* t {number}. (x : t) : string =
    run_target \|
        | Rust => \ () => $'std::rc::Rc::<str>::from((!x).to_string())' : string
        | _ => \ () => format x

:= char_text (code : u8) : string =
    run_target \|
        | Rust => \ () => $'std::rc::Rc::<str>::from(char::from(!code).to_string())' : string
        | Fsharp => \ () => $'string (char !code)' : string
        | _ => \ () => no_arm "sm'.char_text"

:= bytes_between (start : i32) (stop : i32) (text : string) : string =
    run_target \|
        | Rust => \ () =>
            $'std::rc::Rc::<str>::from((!text).get(!start as usize..!stop as usize).unwrap_or(""))' : string
        | Fsharp => \ () =>
            $'(let b = System.Text.Encoding.UTF8.GetBytes(!text : string) in \
                let edge i = i = b.Length || (b.[i] &&& 0xC0uy) <> 0x80uy in \
                if !start < 0 || !start > !stop || !stop > b.Length || not (edge !start) || not (edge !stop) then "" \
                else System.Text.Encoding.UTF8.GetString(b, !start, !stop - !start))' : string
        | _ => \ () => no_arm "sm'.bytes_between"

#### bytes_between_slices_utf8_bytes_or_gives_empty (test)

In [74]:
///- --test --symbol
///> fsharp
///> rust

"aéb" |> bytes_between 1i32 3i32 |> _assert_eq "é"
"aéb" |> bytes_between 3i32 4i32 |> _assert_eq "b"
"aéb" |> bytes_between 1i32 2i32 |> _assert_eq ""
"aéb" |> bytes_between 0i32 9i32 |> _assert_eq ""
"aéb" |> bytes_between (-1i32) 2i32 |> _assert_eq ""
"aéb" |> bytes_between 2i32 1i32 |> _assert_eq ""

.fsx (fsharp)
__assert_eq { expected = é }
__assert_eq { expected = b }
__assert_eq { expected =  }
__assert_eq { expected =  }
__assert_eq { expected =  }
__assert_eq { expected =  }
.rs (rust)
__assert_eq { expected = é }
__assert_eq { expected = b }
__assert_eq { expected =  }
__assert_eq { expected =  }
__assert_eq { expected =  }
__assert_eq { expected =  }

0

In [ ]:
:= all_digits (text : string) : bool =
    run_target \|
        | Rust => \ () => $'\!!text.is_empty() && !text.bytes().all(|byte| byte.is_ascii_digit())' : bool
        | Fsharp => \ () => $'(!text <> "" && String.forall (fun c -> c >= \'0\' && c <= \'9\') !text)' : bool
        | _ => \ () => no_arm "sm'.all_digits"

#### all_digits_accepts_only_nonempty_ascii_digit_runs (test)

In [75]:
///- --test --symbol
///> fsharp
///> rust

"0123456789" |> all_digits |> _assert_eq !+
"" |> all_digits |> _assert_eq !-
"12a" |> all_digits |> _assert_eq !-
"١٢" |> all_digits |> _assert_eq !-

.fsx (fsharp)
__assert_eq { expected = true }
__assert_eq { expected = false }
__assert_eq { expected = false }
__assert_eq { expected = false }
.rs (rust)
__assert_eq { expected = true }
__assert_eq { expected = false }
__assert_eq { expected = false }
__assert_eq { expected = false }

0

#### decimal_text_and_char_text_render_numbers_and_control_characters (test)

In [76]:
///- --test --symbol
///> rust

42i32 |> decimal_text |> _assert_eq "42"
18446744073709551615u64 |> decimal_text |> _assert_eq "18446744073709551615"
31u8 |> char_text |> length |> _assert_eq 1i32
65u8 |> char_text |> _assert_eq "A"

__assert_eq { expected = 42 }
__assert_eq { expected = 18446744073709551615 }
__assert_eq { expected = 1 }
__assert_eq { expected = A }

0

In [ ]:
:= first_line (text : string) : string =
    run_target \|
        | Rust => \ () => $'std::rc::Rc::<str>::from((!text).split(char::from(10u8)).next().unwrap_or(""))' : string
        | Fsharp => \ () =>
            := at = $'(!text).IndexOf(char 10)' : i32
            ? at < 0i32 ?> text |: $'(!text).Substring(0, !at)' : string
        | _ => \ () =>
            := at = index_of "\n" text
            ? at < 0 ?> text |: text |> slice 0 (at - 1)

:= after_first_line (text : string) : string =
    run_target \|
        | Rust => \ () =>
            $'std::rc::Rc::<str>::from((!text).split_once(char::from(10u8)).map(|(_, rest)| rest).unwrap_or(""))' : string
        | Fsharp => \ () =>
            := at = $'(!text).IndexOf(char 10)' : i32
            := rest = at + 1i32
            ? at < 0i32 ?> "" |: $'(!text).Substring(!rest)' : string
        | _ => \ () =>
            := at = index_of "\n" text
            ? at < 0 ?> "" |: text |> slice (at + 1) (length text - 1)

#### first_line_and_after_first_line_split_at_the_first_newline (test)

In [77]:
///- --test --symbol
///> fsharp
///> rust

"é1\nb\nc" |> first_line |> _assert_eq "é1"
"é1\nb\nc" |> after_first_line |> _assert_eq "b\nc"
"only" |> first_line |> _assert_eq "only"
"only" |> after_first_line |> _assert_eq ""
"end\n" |> after_first_line |> _assert_eq ""
"\nx" |> first_line |> _assert_eq ""

.fsx (fsharp)
__assert_eq { expected = é1 }
__assert_eq { expected = b
c }
__assert_eq { expected = only }
__assert_eq { expected =  }
__assert_eq { expected =  }
__assert_eq { expected =  }
.rs (rust)
__assert_eq { expected = é1 }
__assert_eq { expected = b
c }
__assert_eq { expected = only }
__assert_eq { expected =  }
__assert_eq { expected =  }
__assert_eq { expected =  }

0

In [ ]:
:= split_nth (separator : u8) (index : i32) (text : string) : string =
    run_target \|
        | Rust => \ () =>
            $'std::rc::Rc::<str>::from(!text.split(char::from(!separator)).nth(!index as usize).unwrap_or(""))' : string
        | Fsharp => \ () =>
            := parts = $'(!text).Split(char !separator)' : array string
            ? index >= 0i32 && index < am'::length parts ?> am'::index index parts |: ""
        | _ => \ () => no_arm "sm'.split_nth"

#### split_nth_picks_one_field_or_nothing (test)

In [78]:
///- --test --symbol
///> rust

:= ~row = "a" ++# char_text 30u8 ++# "bé" ++# char_text 30u8 ++# "c"
(
    row |> split_nth 30u8 1i32 |> _assert_eq "bé",
    row |> split_nth 30u8 2i32 |> _assert_eq "c",
    row |> split_nth 30u8 3i32 |> _assert_eq ""
)

__assert_eq { expected = bé }
__assert_eq { expected = c }
__assert_eq { expected =  }

0

In [ ]:
:~ ellipsis_end (max : i64) (s : string) =
    := len = s |> length
    ? len <= max
    ?> s
    |:
        := half = (max |> convert::f64) / 2
        := start_half = half |> math::ceil |> convert::i64
        := end_half = half |> math::floor |> convert::i64
        := start = s |> slice 0 (start_half - 1)
        := end = s |> slice (len - end_half) (len - 1)
        [ start; "..."; end ]
        |> concat_list ""

#### ellipsis_end (test)

In [79]:
///- --test --symbol
///> gleam
///> lua
///> fsharp

"12345"
|> ellipsis_end 2
|> _assert_eq "1...5"

"12345"
|> ellipsis_end 3
|> _assert_eq "12...5"

"1234567"
|> ellipsis_end 4
|> _assert_eq "12...67"

.gleam (gleam)
__assert_eq { expected = 1...5 }
__assert_eq { expected = 12...5 }
__assert_eq { expected = 12...67 }
.lua (lua)
__assert_eq { expected = 1...5 }
__assert_eq { expected = 12...5 }
__assert_eq { expected = 12...67 }
.fsx (fsharp)
__assert_eq { expected = 1...5 }
__assert_eq { expected = 12...5 }
__assert_eq { expected = 12...67 }

In [ ]:
:= format_ellipsis s =
    s
    |> format_debug
    |> ellipsis_end 400

:~ replace_regex (pattern : string) (replacement : string) (s : string) : string =
    run_target_args (\ () => s, pattern, replacement) \|
        | Fsharp => \ s, pattern, replacement =>
            $'System.Text.RegularExpressions.Regex.Replace (!s, !pattern, !replacement)'
        | Rust => \ s, pattern, replacement =>
            := regex = pattern |> new_regex |> resultm::unwrap'
            := s = ~~ s
            !\\((regex, s, replacement), $'$"$0.replace_all(&*$1, &*$2)"')
            |> cow_to_std_string
            |> from_std_string
        | Gleam => \ s, pattern, replacement =>
            global "import gleam/regexp"
            $'let assert Ok(re) = !pattern |> regexp.from_string'
            $'re |> regexp.replace(!s, !replacement)' : string
        | _ => \ _ => ~~ ""

#### replace_regex (test)

In [80]:
///- --test --symbol
///> gleam -d gleam_regexp=">= 1.1.0 and < 2.0.0"
///> rust -d regex

" 123"
|> replace_regex "\\s\\w2" ""
|> _assert_eq "3"

.gleam (gleam)
__assert_eq { expected = 3 }
.rs (rust)
__assert_eq { expected = 3 }

In [81]:
///- --test --symbol
///> rust -d regex

"""
        let main args =
            ()

    """
|> replace_regex "(?P<a> *)(?P<b>let\\s+main\\s+.*?\\s*=)" "$a[<EntryPoint>]\n$a$b"
|> _assert_eq """
        [<EntryPoint>]
        let main args =
            ()

    """

__assert_eq { expected =     [<EntryPoint>]
    let main args =
        ()
 }

## main

In [ ]:
///> fsharp

:= main () =
    $'let contains x = !contains x' : ()
    $'let ends_with x = !ends_with x' : ()
    $'let pad_left x = !pad_left x' : ()
    $'let pad_right x = !pad_right x' : ()
    $'let replace x = !replace x' : ()
    $'let replace_regex x = !replace_regex x' : ()
    := slice (a : i32) (b : i32) c = slice a b c
    $'let slice x = !slice x' : ()
    $'let split x = !split x' : ()
    $'let split_string x = !split_string x' : ()
    $'let starts_with x = !starts_with x' : ()
    $'let substring x = !substring x' : ()
    $'let to_lower x = !to_lower x' : ()
    $'let to_upper x = !to_upper x' : ()
    $'let trim x = !trim x' : ()
    $'let ellipsis x = !ellipsis x' : ()
    $'let ellipsis_end x = !ellipsis_end x' : ()
    $'let format_exception x = !format_exception x' : ()
    $'let concat_array x = !concat_array x' : ()
    := concat a (b : seq::seq' string) = concat a b
    $'let concat x = !concat x' : ()
    $'let join\' x = !join' x' : ()
    $'let to_char_array x = !to_char_array x' : ()

## rust

### to_string std_string

In [ ]:
:+ to_string std_string = from_std_string

:~ split_list (separator : string) (s : string) : list string =
    ? length separator = 0i32 ?> [ s ]
    |: arraym::fold (\ acc (part : string) => part :: acc) [] (split separator s |> array) |> listm::rev

:= split_joined (separator : string) (s : string) : string = split_list separator s |> concat_list "|"

#### split_list_uses_the_whole_separator (test)

In [82]:
///- --test --symbol
///> rust
///> gleam

:= ~s = "a::b::::c"
(split_joined "::" s, split_joined "," "x", split_joined "," "a,") |> _assert_eq ("a|b||c", "x", "a|")

.rs (rust)
__assert_eq { expected = (a|b||c, x, a|) }
.gleam (gleam)
__assert_eq { expected = (a|b||c, x, a|) }

0

### decimal_i32

In [83]:
:~ ^ decimal_from (text : string) (i : i32) (n : i32) : option i32 =
    ? i >= length text ?> Some n
    |:
        := d : i32 = index_of (substring i 1i32 text) "0123456789"
        := overflows = n > 214748364i32 || (n = 214748364i32 && d > 7i32)
        ? d < 0i32 || overflows ?> None |: decimal_from text (i + 1i32) (n * 10i32 + d)

:= decimal_i32 (text : string) : option i32 = ? text = "" ?> None |: decimal_from text 0i32 0i32

#### decimal_text_is_strict (test)

In [84]:
///- --test --symbol
///> rust
///> gleam

:= ~t = "2147483647"
(decimal_i32 t, decimal_i32 "12", decimal_i32 "", decimal_i32 "-1", decimal_i32 "1x", decimal_i32 "2147483648")
|> _assert_eq (Some 2147483647i32, Some 12i32, None, None, None, None)

.rs (rust)
__assert_eq { expected = (Some(2147483647), Some(12), None, None, None, None) }
.gleam (gleam)
__assert_eq { expected = (Some(2147483647), Some(12), None, None, None, None) }

0

#### default_value_inside_a_fold_closure_keeps_the_captured_seed (test)

In [85]:
///- --test --symbol
///> rust -d regex

:= pattern = new_regex "(?<a>x+)" |> resultm::unwrap'
regex_captures "yxx" pattern
|> am'::vec_fold' \ found captures =>
        ? found <> "" ?> found |: optionm'::default_value' "" (map::get "a" captures)
    ""
|> _assert_eq "xx"

__assert_eq { expected = xx }